# 02 - Data Preparation

## Football Analytics Platform

### Purpose
This notebook prepares StatsBomb Events and 360 data for player-performance
assessment and machine-learning modelling.

### Main Tasks
- Load the required StatsBomb data
- Integrate Events and 360 datasets
- Handle missing spatial information
- Extract information from nested JSON structures
- Engineer passing and progression features
- Engineer spatial pressure features
- Engineer shooting and defensive features
- Aggregate event-level features to player-match level
- Create historical player baselines
- Produce the final modelling dataset

In [141]:
import pandas as pd
import numpy as np
import json
import math

from pathlib import Path

In [142]:
base_path = Path("open-data-master/data")

events_folder = base_path / "events"
three_sixty_folder = base_path / "three-sixty"

In [143]:
print("Events folder exists:", events_folder.exists())
print("360 folder exists:", three_sixty_folder.exists())

Events folder exists: True
360 folder exists: True


In [144]:
three_sixty_files = list(three_sixty_folder.glob("*.json"))

match_ids_360 = [
    file.stem
    for file in three_sixty_files
]

print("Matches with 360 data:", len(match_ids_360))
print(match_ids_360[:10])

Matches with 360 data: 426
['3764440', '3764661', '3773369', '3773372', '3773377', '3773386', '3773387', '3773403', '3773415', '3773428']


In [145]:
valid_match_ids = [
    match_id
    for match_id in match_ids_360
    if (events_folder / f"{match_id}.json").exists()
]

print("Matches with both Events and 360:", len(valid_match_ids))

Matches with both Events and 360: 426


In [146]:
def load_match(match_id):

    
    events_file = events_folder / f"{match_id}.json"
    three_sixty_file = three_sixty_folder / f"{match_id}.json"

    # Load Events data
    with open(events_file, "r", encoding="utf-8") as file:
        events_data = json.load(file)

    # Load 360 data
    with open(three_sixty_file, "r", encoding="utf-8") as file:
        data_360 = json.load(file)

    # Convert to DataFrames
    events_df = pd.DataFrame(events_data)
    df_360 = pd.DataFrame(data_360)

    # Merge Events + 360
    merged_df = events_df.merge(
        df_360[["event_uuid", "freeze_frame", "visible_area"]],
        left_on="id",
        right_on="event_uuid",
        how="left"
    )

    # Add match ID
    merged_df["match_id"] = match_id

    return merged_df

In [147]:
test_match = load_match("3764440")

print("Shape:", test_match.shape)

Shape: (4160, 40)


In [148]:
test_match[
    [
        "id",
        "type",
        "freeze_frame",
        "match_id"
    ]
].head()

,id,type,freeze_frame,match_id
0,d130c53e-c291-48bc-8372-721330fb160b,"{'id': 35, 'name': 'Starting XI'}",NaN,3764440
1,d5c25be3-487e-40a0-887d-f5d274d2d57b,"{'id': 35, 'name': 'Starting XI'}",NaN,3764440
2,900ca195-c93a-4143-874e-ce2ad56c3752,"{'id': 18, 'name': 'Half Start'}",NaN,3764440
3,0ce64a65-8a8b-479b-bdc4-eaec24d8423d,"{'id': 18, 'name': 'Half Start'}",NaN,3764440
4,23743e50-fbe4-4929-938e-9b3ab39ff4ff,"{'id': 30, 'name': 'Pass'}","[{'teammate': False, 'actor': False, 'keeper':...",3764440


In [149]:
test_match["event_type"] = test_match["type"].apply(
    lambda x: x.get("name") if isinstance(x, dict) else x
)

In [150]:
test_match[
    ["type", "event_type"]
].head(10)

,type,event_type
0,"{'id': 35, 'name': 'Starting XI'}",Starting XI
1,"{'id': 35, 'name': 'Starting XI'}",Starting XI
2,"{'id': 18, 'name': 'Half Start'}",Half Start
3,"{'id': 18, 'name': 'Half Start'}",Half Start
4,"{'id': 30, 'name': 'Pass'}",Pass
5,"{'id': 42, 'name': 'Ball Receipt*'}",Ball Receipt*
6,"{'id': 43, 'name': 'Carry'}",Carry
7,"{'id': 30, 'name': 'Pass'}",Pass
8,"{'id': 42, 'name': 'Ball Receipt*'}",Ball Receipt*
9,"{'id': 43, 'name': 'Carry'}",Carry


In [151]:
def get_nearest_opponent(freeze_frame):

    if not isinstance(freeze_frame, list):
        return None, None

    actor = next(
        (
            player for player in freeze_frame
            if player.get("actor") == True
        ),
        None
    )

    if actor is None:
        return None, None

    actor_location = actor.get("location")

    if actor_location is None:
        return None, None

    opponents = [
        player for player in freeze_frame
        if player.get("teammate") == False
        and player.get("location") is not None
    ]

    if len(opponents) == 0:
        return None, None

    distances = []

    for opponent in opponents:

        opponent_location = opponent["location"]

        distance = math.sqrt(
            (opponent_location[0] - actor_location[0]) ** 2 +
            (opponent_location[1] - actor_location[1]) ** 2
        )

        distances.append(distance)

    nearest_index = distances.index(min(distances))

    nearest_opponent = opponents[nearest_index]

    return (
        nearest_opponent["location"],
        distances[nearest_index]
    )

In [152]:
print(test_match.columns.tolist())

['id', 'index', 'period', 'timestamp', 'minute', 'second', 'type', 'possession', 'possession_team', 'play_pattern', 'team', 'duration', 'tactics', 'related_events', 'player', 'position', 'location', 'pass', 'carry', 'under_pressure', 'dribble', 'shot', 'goalkeeper', 'out', 'ball_recovery', 'duel', 'ball_receipt', 'clearance', 'off_camera', 'counterpress', 'interception', 'foul_won', 'foul_committed', 'substitution', '50_50', 'injury_stoppage', 'event_uuid', 'freeze_frame', 'visible_area', 'match_id', 'event_type']


In [153]:
nearest_results = test_match["freeze_frame"].apply(
    get_nearest_opponent
)

In [154]:
nearest_df = pd.DataFrame(
    nearest_results.tolist(),
    index=test_match.index,
    columns=[
        "nearest_opponent_location",
        "nearest_opponent_distance"
    ]
)

In [155]:
test_match[
    ["nearest_opponent_location", "nearest_opponent_distance"]
] = nearest_df

In [156]:
print("nearest_opponent_location" in test_match.columns)
print("nearest_opponent_distance" in test_match.columns)

True
True


In [157]:
test_match[
    [
        "event_type",
        "nearest_opponent_location",
        "nearest_opponent_distance"
    ]
].dropna().head(10)

,event_type,nearest_opponent_location,nearest_opponent_distance
4,Pass,"[52.273950315181956, 38.42847753920154]",8.884702
5,Ball Receipt*,"[58.525749040615366, 55.616830930614796]",10.734909
6,Carry,"[58.525749040615366, 55.616830930614796]",10.734909
7,Pass,"[58.99838057189282, 51.40565894914739]",8.898384
8,Ball Receipt*,"[55.378889386232515, 48.42932763485586]",22.101975
9,Carry,"[55.378889386232515, 48.42932763485586]",22.101975
10,Pass,"[55.09060399419812, 49.287765588342715]",21.169590
11,Ball Receipt*,"[47.39650916051406, 28.33798319316506]",15.873415
12,Carry,"[47.39650916051406, 28.33798319316506]",15.873415
13,Pass,"[42.95139938794394, 27.757633866350695]",9.798178


In [158]:
def count_nearby_opponents(freeze_frame, radius=5):

    if not isinstance(freeze_frame, list):
        return None

    actor = next(
        (
            player for player in freeze_frame
            if player.get("actor") == True
        ),
        None
    )

    if actor is None:
        return None

    actor_location = actor.get("location")

    if actor_location is None:
        return None

    opponents = [
        player for player in freeze_frame
        if player.get("teammate") == False
        and player.get("location") is not None
    ]

    count = 0

    for opponent in opponents:

        opponent_location = opponent["location"]

        distance = math.sqrt(
            (opponent_location[0] - actor_location[0]) ** 2 +
            (opponent_location[1] - actor_location[1]) ** 2
        )

        if distance <= radius:
            count += 1

    return count

In [159]:
test_match["nearby_opponents_5"] = test_match[
    "freeze_frame"
].apply(count_nearby_opponents)

In [160]:
test_match[
    [
        "event_type",
        "nearest_opponent_distance",
        "nearby_opponents_5"
    ]
].dropna().head(20)

,event_type,nearest_opponent_distance,nearby_opponents_5
4,Pass,8.884702,0.0
5,Ball Receipt*,10.734909,0.0
6,Carry,10.734909,0.0
7,Pass,8.898384,0.0
8,Ball Receipt*,22.101975,0.0
9,Carry,22.101975,0.0
10,Pass,21.169590,0.0
11,Ball Receipt*,15.873415,0.0
12,Carry,15.873415,0.0
13,Pass,9.798178,0.0


In [161]:
test_match["nearby_opponents_5"].value_counts(
    dropna=False
).sort_index()

nearby_opponents_5
0.0    1913
1.0    1515
2.0     390
3.0      85
4.0       9
6.0       2
NaN     246
Name: count, dtype: int64

In [162]:
test_match[
    "nearest_opponent_distance"
].describe()

count    3914.000000
mean        6.430309
std         5.169467
min         0.134817
25%         2.592150
50%         4.864531
75%         9.203136
max        47.375719
Name: nearest_opponent_distance, dtype: float64

In [163]:
passes_test = test_match[
    test_match["event_type"] == "Pass"
]

print("Number of passes:", len(passes_test))

print(
    passes_test["nearby_opponents_5"]
    .value_counts()
    .sort_index()
)

Number of passes: 1215
nearby_opponents_5
0.0    499
1.0    478
2.0    145
3.0     27
4.0      1
Name: count, dtype: int64


In [164]:
passes_test[
    "nearest_opponent_distance"
].describe()            

count    1150.000000
mean        5.824588
std         4.972044
min         0.289843
25%         2.528544
50%         4.272359
75%         7.647979
max        47.375719
Name: nearest_opponent_distance, dtype: float64

In [165]:
test_match["under_spatial_pressure"] = pd.NA

has_distance = test_match[
    "nearest_opponent_distance"
].notna()

test_match.loc[
    has_distance,
    "under_spatial_pressure"
] = (
    test_match.loc[
        has_distance,
        "nearest_opponent_distance"
    ] <= 5
)

In [166]:
passes_test = test_match[
    test_match["event_type"] == "Pass"
].copy()

In [167]:
print(
    passes_test[
        "under_spatial_pressure"
    ].value_counts(dropna=False)
)

under_spatial_pressure
True     651
False    499
<NA>      65
Name: count, dtype: int64


In [168]:
def is_pass_completed(pass_data):

    if not isinstance(pass_data, dict):
        return None

    outcome = pass_data.get("outcome")

    if outcome is None:
        return True
    else:
        return False

In [169]:
passes_test["pass_completed"] = passes_test["pass"].apply(
    is_pass_completed
)

In [170]:
passes_test[
    [
        "player",
        "under_spatial_pressure",
        "nearest_opponent_distance",
        "pass_completed"
    ]
].head(20)

,player,under_spatial_pressure,nearest_opponent_distance,pass_completed
4,"{'id': 12072, 'name': 'Pere Milla Peña'}",False,8.884702,True
7,"{'id': 24517, 'name': 'José Raúl Gutiérrez Par...",False,8.898384,True
10,"{'id': 24169, 'name': 'Gonzalo Cacicedo Verdú'}",False,21.169590,True
13,"{'id': 9857, 'name': 'José Manuel Sánchez Guil...",False,9.798178,True
16,"{'id': 24169, 'name': 'Gonzalo Cacicedo Verdú'}",True,2.008600,True
21,"{'id': 20055, 'name': 'Marc-André ter Stegen'}",False,14.375390,True
24,"{'id': 5492, 'name': 'Samuel Yves Umtiti'}",False,10.736957,True
27,"{'id': 5213, 'name': 'Gerard Piqué Bernabéu'}",False,13.143221,True
30,"{'id': 8118, 'name': 'Frenkie de Jong'}",True,4.436296,True
33,"{'id': 5492, 'name': 'Samuel Yves Umtiti'}",False,7.762973,True


In [171]:
pressure_completion = (
    passes_test
    .dropna(subset=["under_spatial_pressure"])
    .groupby("under_spatial_pressure")["pass_completed"]
    .mean()
    * 100
)

print(pressure_completion)

under_spatial_pressure
False    92.585170
True     86.328725
Name: pass_completed, dtype: float64


In [172]:
passes_test["player_name"] = passes_test["player"].apply(
    lambda x: x.get("name") if isinstance(x, dict) else x
)

In [173]:
player_pressure_stats = (
    passes_test
    .dropna(subset=["under_spatial_pressure"])
    .groupby(
        ["player_name", "under_spatial_pressure"]
    )["pass_completed"]
    .agg(["count", "mean"])
    .reset_index()
)

In [174]:
player_pressure_stats["completion_percentage"] = (
    player_pressure_stats["mean"] * 100
)

In [175]:
player_pressure_stats.head(20)

,player_name,under_spatial_pressure,count,mean,completion_percentage
0,Antoine Griezmann,False,2,1.000000,100.000000
1,Antoine Griezmann,True,5,0.800000,80.000000
2,Antonio Barragán Fernández,False,14,0.857143,85.714286
3,Antonio Barragán Fernández,True,26,0.923077,92.307692
4,Clément Lenglet,False,10,0.900000,90.000000
5,Clément Lenglet,True,4,1.000000,100.000000
6,Edgar Badía Guardiola,False,24,0.875000,87.500000
7,Edgar Badía Guardiola,True,5,0.200000,20.000000
8,Emiliano Ariel Rigoni,False,4,0.500000,50.000000
9,Emiliano Ariel Rigoni,True,18,0.722222,72.222222


In [176]:
completion_table = player_pressure_stats.pivot(
    index="player_name",
    columns="under_spatial_pressure",
    values="completion_percentage"
)

completion_table.columns = [
    "no_pressure_completion",
    "pressure_completion"
]

completion_table.head()

,no_pressure_completion,pressure_completion
player_name,,
Antoine Griezmann,100.000000,80.000000
Antonio Barragán Fernández,85.714286,92.307692
Clément Lenglet,90.000000,100.000000
Edgar Badía Guardiola,87.500000,20.000000
Emiliano Ariel Rigoni,50.000000,72.222222


In [177]:
count_table = player_pressure_stats.pivot(
    index="player_name",
    columns="under_spatial_pressure",
    values="count"
)

count_table.columns = [
    "no_pressure_passes",
    "pressure_passes"
]

count_table.head()

,no_pressure_passes,pressure_passes
player_name,,
Antoine Griezmann,2,5
Antonio Barragán Fernández,14,26
Clément Lenglet,10,4
Edgar Badía Guardiola,24,5
Emiliano Ariel Rigoni,4,18


In [178]:
player_pass_summary = (
    passes_test
    .dropna(subset=["under_spatial_pressure"])
    .groupby(
        ["player_name", "under_spatial_pressure"]
    )
    .agg(
        pass_attempts=("pass_completed", "count"),
        passes_completed=("pass_completed", "sum"),
        completion_rate=("pass_completed", "mean")
    )
    .reset_index()
)

In [179]:
player_pass_summary["completion_rate"] = (
    player_pass_summary["completion_rate"] * 100
)

In [180]:
player_pass_comparison = player_pass_summary.pivot(
    index="player_name",
    columns="under_spatial_pressure",
    values=[
        "pass_attempts",
        "passes_completed",
        "completion_rate"
    ]
)

In [181]:
player_pass_comparison.columns = [
    "no_pressure_attempts",
    "pressure_attempts",
    "no_pressure_completed",
    "pressure_completed",
    "no_pressure_completion_rate",
    "pressure_completion_rate"
]

In [182]:
player_pass_comparison = (
    player_pass_comparison.reset_index()
)

In [183]:
player_pass_comparison["pressure_difference"] = (
    player_pass_comparison["pressure_completion_rate"]
    - player_pass_comparison["no_pressure_completion_rate"]
)

In [184]:
player_pass_comparison.head(20)

,player_name,no_pressure_attempts,pressure_attempts,no_pressure_completed,pressure_completed,no_pressure_completion_rate,pressure_completion_rate,pressure_difference
0,Antoine Griezmann,2.0,5.0,2.0,4.0,100.000000,80.000000,-20.000000
1,Antonio Barragán Fernández,14.0,26.0,12.0,24.0,85.714286,92.307692,6.593407
2,Clément Lenglet,10.0,4.0,9.0,4.0,90.000000,100.000000,10.000000
3,Edgar Badía Guardiola,24.0,5.0,21.0,1.0,87.500000,20.000000,-67.500000
4,Emiliano Ariel Rigoni,4.0,18.0,2.0,13.0,50.000000,72.222222,22.222222
5,Fidel Chaves de la Torre,2.0,13.0,1.0,10.0,50.000000,76.923077,26.923077
6,Francisco António Machado Mota de Castro Trincão,5.0,31.0,5.0,29.0,100.000000,93.548387,-6.451613
7,Frenkie de Jong,49.0,51.0,46.0,49.0,93.877551,96.078431,2.200880
8,Gerard Piqué Bernabéu,61.0,29.0,59.0,24.0,96.721311,82.758621,-13.962691
9,Gonzalo Cacicedo Verdú,33.0,16.0,33.0,16.0,100.000000,100.000000,0.000000


In [185]:
percentage_columns = [
    "no_pressure_completion_rate",
    "pressure_completion_rate",
    "pressure_difference"
]

player_pass_comparison[percentage_columns] = (
    player_pass_comparison[percentage_columns].round(2)
)

In [186]:
player_pass_comparison.head(20)

,player_name,no_pressure_attempts,pressure_attempts,no_pressure_completed,pressure_completed,no_pressure_completion_rate,pressure_completion_rate,pressure_difference
0,Antoine Griezmann,2.0,5.0,2.0,4.0,100.00,80.00,-20.00
1,Antonio Barragán Fernández,14.0,26.0,12.0,24.0,85.71,92.31,6.59
2,Clément Lenglet,10.0,4.0,9.0,4.0,90.00,100.00,10.00
3,Edgar Badía Guardiola,24.0,5.0,21.0,1.0,87.50,20.00,-67.50
4,Emiliano Ariel Rigoni,4.0,18.0,2.0,13.0,50.00,72.22,22.22
5,Fidel Chaves de la Torre,2.0,13.0,1.0,10.0,50.00,76.92,26.92
6,Francisco António Machado Mota de Castro Trincão,5.0,31.0,5.0,29.0,100.00,93.55,-6.45
7,Frenkie de Jong,49.0,51.0,46.0,49.0,93.88,96.08,2.20
8,Gerard Piqué Bernabéu,61.0,29.0,59.0,24.0,96.72,82.76,-13.96
9,Gonzalo Cacicedo Verdú,33.0,16.0,33.0,16.0,100.00,100.00,0.00


In [187]:
player_nearby_stats = (
    passes_test
    .dropna(subset=["nearby_opponents_5"])
    .groupby("player_name")
    .agg(
        total_passes_with_360=("nearby_opponents_5", "count"),
        average_nearby_opponents=("nearby_opponents_5", "mean"),
        max_nearby_opponents=("nearby_opponents_5", "max"),
        average_nearest_opponent_distance=(
            "nearest_opponent_distance",
            "mean"
        )
    )
    .reset_index()
)

In [188]:
player_nearby_stats[
    [
        "average_nearby_opponents",
        "average_nearest_opponent_distance"
    ]
] = player_nearby_stats[
    [
        "average_nearby_opponents",
        "average_nearest_opponent_distance"
    ]
].round(2)

In [189]:
player_nearby_stats.head(20)

,player_name,total_passes_with_360,average_nearby_opponents,max_nearby_opponents,average_nearest_opponent_distance
0,Antoine Griezmann,7,1.00,2.0,4.39
1,Antonio Barragán Fernández,40,0.78,2.0,4.96
2,Clément Lenglet,14,0.29,1.0,8.09
3,Edgar Badía Guardiola,29,0.17,1.0,16.71
4,Emiliano Ariel Rigoni,22,1.18,2.0,5.75
5,Fidel Chaves de la Torre,15,1.40,2.0,2.49
6,Francisco António Machado Mota de Castro Trincão,36,1.22,3.0,3.36
7,Frenkie de Jong,100,0.71,3.0,5.70
8,Gerard Piqué Bernabéu,90,0.39,3.0,7.29
9,Gonzalo Cacicedo Verdú,49,0.33,1.0,9.43


In [190]:
def prepare_match(match_id):

    # 1. Load and merge Events + 360
    match_df = load_match(match_id)

    # 2. Extract clean event type
    match_df["event_type"] = match_df["type"].apply(
        lambda x: x.get("name") if isinstance(x, dict) else x
    )

    # 3. Extract clean player name
    match_df["player_name"] = match_df["player"].apply(
        lambda x: x.get("name") if isinstance(x, dict) else x
    )

    # 4. Calculate nearest opponent
    nearest_results = match_df["freeze_frame"].apply(
        get_nearest_opponent
    )

    nearest_df = pd.DataFrame(
        nearest_results.tolist(),
        index=match_df.index,
        columns=[
            "nearest_opponent_location",
            "nearest_opponent_distance"
        ]
    )

    match_df[
        ["nearest_opponent_location", "nearest_opponent_distance"]
    ] = nearest_df

    # 5. Count opponents within 5 units
    match_df["nearby_opponents_5"] = (
        match_df["freeze_frame"]
        .apply(count_nearby_opponents)
    )

    # 6. Create spatial-pressure feature
    match_df["under_spatial_pressure"] = pd.NA

    has_distance = match_df[
        "nearest_opponent_distance"
    ].notna()

    match_df.loc[
        has_distance,
        "under_spatial_pressure"
    ] = (
        match_df.loc[
            has_distance,
            "nearest_opponent_distance"
        ] <= 5
    )

    # 7. Calculate pass completion
    match_df["pass_completed"] = match_df["pass"].apply(
        lambda x:
            True if isinstance(x, dict) and x.get("outcome") is None
            else False if isinstance(x, dict)
            else pd.NA
    )

    return match_df

In [191]:
prepared_test_match = prepare_match("3764440")

In [192]:
prepared_test_match[
    [
        "match_id",
        "player_name",
        "event_type",
        "nearest_opponent_distance",
        "nearby_opponents_5",
        "under_spatial_pressure",
        "pass_completed"
    ]
].head(20)

,match_id,player_name,event_type,nearest_opponent_distance,nearby_opponents_5,under_spatial_pressure,pass_completed
0,3764440,NaN,Starting XI,NaN,NaN,<NA>,<NA>
1,3764440,NaN,Starting XI,NaN,NaN,<NA>,<NA>
2,3764440,NaN,Half Start,NaN,NaN,<NA>,<NA>
3,3764440,NaN,Half Start,NaN,NaN,<NA>,<NA>
4,3764440,Pere Milla Peña,Pass,8.884702,0.0,False,True
5,3764440,José Raúl Gutiérrez Parejo,Ball Receipt*,10.734909,0.0,False,<NA>
6,3764440,José Raúl Gutiérrez Parejo,Carry,10.734909,0.0,False,<NA>
7,3764440,José Raúl Gutiérrez Parejo,Pass,8.898384,0.0,False,True
8,3764440,Gonzalo Cacicedo Verdú,Ball Receipt*,22.101975,0.0,False,<NA>
9,3764440,Gonzalo Cacicedo Verdú,Carry,22.101975,0.0,False,<NA>


In [193]:
required_columns = [
    "player_name",
    "event_type",
    "nearest_opponent_distance",
    "nearby_opponents_5",
    "under_spatial_pressure",
    "pass_completed"
]

for column in required_columns:
    print(column, column in prepared_test_match.columns)

player_name True
event_type True
nearest_opponent_distance True
nearby_opponents_5 True
under_spatial_pressure True
pass_completed True


In [194]:
prepared_matches = []

for match_id in valid_match_ids:
    try:
        prepared = prepare_match(match_id)
        prepared_matches.append(prepared)

    except Exception as e:
        print(f"Problem with match {match_id}: {e}")

Problem with match 3845506: Expecting ',' delimiter: line 92794 column 3 (char 2637824)


In [195]:
match_id = "3845506"

events_file = events_folder / f"{match_id}.json"
three_sixty_file = three_sixty_folder / f"{match_id}.json"

for name, path in [
    ("Events", events_file),
    ("360", three_sixty_file)
]:
    try:
        with open(path, "r", encoding="utf-8") as file:
            json.load(file)

        print(f"{name}: OK")

    except json.JSONDecodeError as e:
        print(f"{name}: JSON ERROR")
        print(e)

Events: OK
360: JSON ERROR
Expecting ',' delimiter: line 92794 column 3 (char 2637824)


In [196]:
print("Successful matches:", len(prepared_matches))

Successful matches: 425


In [197]:
all_360_events = pd.concat(
    prepared_matches,
    ignore_index=True
)

In [198]:
print("Total event rows:", len(all_360_events))
print("Shape:", all_360_events.shape)

Total event rows: 1582258
Shape: (1582258, 52)


In [199]:
all_360_events.to_pickle(
    "all_360_prepared_events.pkl"
)

In [200]:
from pathlib import Path

events_folder = Path("open-data-master/data/events")
three_sixty_folder = Path("open-data-master/data/three-sixty")

event_files = list(events_folder.glob("*.json"))
three_sixty_files = list(three_sixty_folder.glob("*.json"))

total_event_matches = len(event_files)
total_360_matches = len(three_sixty_files)

coverage_percentage = (
    total_360_matches / total_event_matches
) * 100

print("Total matches with Events data:", total_event_matches)
print("Total matches with 360 data:", total_360_matches)
print("360 match coverage:", round(coverage_percentage, 2), "%")

Total matches with Events data: 4235
Total matches with 360 data: 426
360 match coverage: 10.06 %


In [201]:
event_match_ids = {
    file.stem for file in event_files
}

three_sixty_match_ids = {
    file.stem for file in three_sixty_files
}

matches_with_both = (
    event_match_ids & three_sixty_match_ids
)

coverage_percentage = (
    len(matches_with_both) /
    len(event_match_ids)
) * 100

print("Total Events matches:", len(event_match_ids))
print("Total 360 matches:", len(three_sixty_match_ids))
print("Matches with both:", len(matches_with_both))

print(
    "Percentage of matches with 360 coverage:",
    round(coverage_percentage, 2),
    "%"
)

Total Events matches: 4235
Total 360 matches: 426
Matches with both: 426
Percentage of matches with 360 coverage: 10.06 %


In [202]:
all_360_events = pd.concat(
    prepared_matches,
    ignore_index=True
)

all_360_events.to_pickle(
    "all_360_prepared_events.pkl"
)

print("Matches processed:", len(prepared_matches))
print("Total event rows:", len(all_360_events))
print("Saved successfully.")

Matches processed: 425
Total event rows: 1582258
Saved successfully.


In [203]:
all_360_events.head()

,id,index,period,timestamp,minute,second,type,possession,possession_team,play_pattern,...,nearest_opponent_location,nearest_opponent_distance,nearby_opponents_5,under_spatial_pressure,pass_completed,block,bad_behaviour,miscontrol,player_off,half_start
0,d130c53e-c291-48bc-8372-721330fb160b,1,1,00:00:00.000,0,0,"{'id': 35, 'name': 'Starting XI'}",1,"{'id': 217, 'name': 'Barcelona'}","{'id': 1, 'name': 'Regular Play'}",...,None,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN
1,d5c25be3-487e-40a0-887d-f5d274d2d57b,2,1,00:00:00.000,0,0,"{'id': 35, 'name': 'Starting XI'}",1,"{'id': 217, 'name': 'Barcelona'}","{'id': 1, 'name': 'Regular Play'}",...,None,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN
2,900ca195-c93a-4143-874e-ce2ad56c3752,3,1,00:00:00.000,0,0,"{'id': 18, 'name': 'Half Start'}",1,"{'id': 217, 'name': 'Barcelona'}","{'id': 1, 'name': 'Regular Play'}",...,None,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN
3,0ce64a65-8a8b-479b-bdc4-eaec24d8423d,4,1,00:00:00.000,0,0,"{'id': 18, 'name': 'Half Start'}",1,"{'id': 217, 'name': 'Barcelona'}","{'id': 1, 'name': 'Regular Play'}",...,None,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN
4,23743e50-fbe4-4929-938e-9b3ab39ff4ff,5,1,00:00:00.438,0,0,"{'id': 30, 'name': 'Pass'}",2,"{'id': 1042, 'name': 'Elche'}","{'id': 9, 'name': 'From Kick Off'}",...,"[52.273950315181956, 38.42847753920154]",8.884702,0.0,False,True,NaN,NaN,NaN,NaN,NaN


In [204]:
all_360_events.shape

(1582258, 52)

In [205]:
all_360_events["position_name"] = (
    all_360_events["position"].apply(
        lambda x: x.get("name") if isinstance(x, dict) else x
    )
)

In [206]:
all_360_events[
    ["player_name", "position", "position_name"]
].head(10)

,player_name,position,position_name
0,NaN,NaN,NaN
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,Pere Milla Peña,"{'id': 17, 'name': 'Right Wing'}",Right Wing
5,José Raúl Gutiérrez Parejo,"{'id': 13, 'name': 'Right Center Midfield'}",Right Center Midfield
6,José Raúl Gutiérrez Parejo,"{'id': 13, 'name': 'Right Center Midfield'}",Right Center Midfield
7,José Raúl Gutiérrez Parejo,"{'id': 13, 'name': 'Right Center Midfield'}",Right Center Midfield
8,Gonzalo Cacicedo Verdú,"{'id': 4, 'name': 'Center Back'}",Center Back
9,Gonzalo Cacicedo Verdú,"{'id': 4, 'name': 'Center Back'}",Center Back


In [207]:
all_360_events["is_pass"] = (
    all_360_events["event_type"] == "Pass"
)

all_360_events["is_shot"] = (
    all_360_events["event_type"] == "Shot"
)

all_360_events["is_carry"] = (
    all_360_events["event_type"] == "Carry"
)

all_360_events["is_miscontrol"] = (
    all_360_events["event_type"] == "Miscontrol"
)

In [208]:
all_360_events[
    [
        "match_id",
        "player_name",
        "event_type",
        "is_pass",
        "is_shot",
        "is_carry",
        "is_miscontrol"
    ]
].head(20)

,match_id,player_name,event_type,is_pass,is_shot,is_carry,is_miscontrol
0,3764440,NaN,Starting XI,False,False,False,False
1,3764440,NaN,Starting XI,False,False,False,False
2,3764440,NaN,Half Start,False,False,False,False
3,3764440,NaN,Half Start,False,False,False,False
4,3764440,Pere Milla Peña,Pass,True,False,False,False
5,3764440,José Raúl Gutiérrez Parejo,Ball Receipt*,False,False,False,False
6,3764440,José Raúl Gutiérrez Parejo,Carry,False,False,True,False
7,3764440,José Raúl Gutiérrez Parejo,Pass,True,False,False,False
8,3764440,Gonzalo Cacicedo Verdú,Ball Receipt*,False,False,False,False
9,3764440,Gonzalo Cacicedo Verdú,Carry,False,False,True,False


In [209]:
print(all_360_events.columns.tolist())

['id', 'index', 'period', 'timestamp', 'minute', 'second', 'type', 'possession', 'possession_team', 'play_pattern', 'team', 'duration', 'tactics', 'related_events', 'player', 'position', 'location', 'pass', 'carry', 'under_pressure', 'dribble', 'shot', 'goalkeeper', 'out', 'ball_recovery', 'duel', 'ball_receipt', 'clearance', 'off_camera', 'counterpress', 'interception', 'foul_won', 'foul_committed', 'substitution', '50_50', 'injury_stoppage', 'event_uuid', 'freeze_frame', 'visible_area', 'match_id', 'event_type', 'player_name', 'nearest_opponent_location', 'nearest_opponent_distance', 'nearby_opponents_5', 'under_spatial_pressure', 'pass_completed', 'block', 'bad_behaviour', 'miscontrol', 'player_off', 'half_start', 'position_name', 'is_pass', 'is_shot', 'is_carry', 'is_miscontrol']


In [210]:
all_360_events["pressured_pass"] = (
    all_360_events["is_pass"]
    & (all_360_events["under_spatial_pressure"] == True)
)

all_360_events["pressured_pass_completed"] = (
    all_360_events["pressured_pass"]
    & (all_360_events["pass_completed"] == True)
)

In [211]:
all_360_events[
    [
        "player_name",
        "event_type",
        "under_spatial_pressure",
        "pass_completed",
        "pressured_pass",
        "pressured_pass_completed"
    ]
][all_360_events["is_pass"]].head(20)

,player_name,event_type,under_spatial_pressure,pass_completed,pressured_pass,pressured_pass_completed
4,Pere Milla Peña,Pass,False,True,False,False
7,José Raúl Gutiérrez Parejo,Pass,False,True,False,False
10,Gonzalo Cacicedo Verdú,Pass,False,True,False,False
13,José Manuel Sánchez Guillén,Pass,False,True,False,False
16,Gonzalo Cacicedo Verdú,Pass,True,True,True,True
21,Marc-André ter Stegen,Pass,False,True,False,False
24,Samuel Yves Umtiti,Pass,False,True,False,False
27,Gerard Piqué Bernabéu,Pass,False,True,False,False
30,Frenkie de Jong,Pass,True,True,True,True
33,Samuel Yves Umtiti,Pass,False,True,False,False


In [212]:
player_match_features = (
    all_360_events
    .dropna(subset=["player_name"])
    .groupby(["match_id", "player_name"])
    .agg(
        total_events=("event_type", "count"),

        pass_attempts=("is_pass", "sum"),
        passes_completed=("pass_completed", "sum"),

        pressured_pass_attempts=("pressured_pass", "sum"),
        pressured_passes_completed=("pressured_pass_completed", "sum"),

        shots=("is_shot", "sum"),
        carries=("is_carry", "sum"),
        miscontrols=("is_miscontrol", "sum"),

        avg_nearest_opponent_distance=(
            "nearest_opponent_distance", "mean"
        ),

        avg_nearby_opponents=(
            "nearby_opponents_5", "mean"
        )
    )
    .reset_index()
)

In [213]:
print(
    player_match_features[
        ["pass_attempts", "passes_completed"]
    ].dtypes
)

pass_attempts        int64
passes_completed    object
dtype: object


In [214]:
player_match_features["passes_completed"] = pd.to_numeric(
    player_match_features["passes_completed"],
    errors="coerce"
)

In [215]:
player_match_features["pass_completion_rate"] = (
    player_match_features["passes_completed"]
    / player_match_features["pass_attempts"]
    * 100
)

In [216]:
print(
    player_match_features[
        ["pass_attempts", "passes_completed"]
    ].dtypes
)

pass_attempts       int64
passes_completed    int64
dtype: object


In [217]:
import numpy as np

player_match_features["pass_completion_rate"] = np.where(
    player_match_features["pass_attempts"] > 0,
    (
        player_match_features["passes_completed"]
        / player_match_features["pass_attempts"]
        * 100
    ),
    np.nan
)

In [218]:
print(
    player_match_features[
        [
            "pressured_pass_attempts",
            "pressured_passes_completed"
        ]
    ].dtypes
)

pressured_pass_attempts       int64
pressured_passes_completed    int64
dtype: object


In [219]:
player_match_features["pressured_pass_completion_rate"] = np.where(
    player_match_features["pressured_pass_attempts"] > 0,
    (
        player_match_features["pressured_passes_completed"]
        / player_match_features["pressured_pass_attempts"]
        * 100
    ),
    np.nan
)

In [220]:
player_match_features[
    [
        "match_id",
        "player_name",
        "pass_attempts",
        "passes_completed",
        "pass_completion_rate",
        "pressured_pass_attempts",
        "pressured_passes_completed",
        "pressured_pass_completion_rate",
        "shots",
        "carries",
        "miscontrols"
    ]
].head(20)

,match_id,player_name,pass_attempts,passes_completed,pass_completion_rate,pressured_pass_attempts,pressured_passes_completed,pressured_pass_completion_rate,shots,carries,miscontrols
0,3764440,Antoine Griezmann,8,7,87.500000,5,4,80.000000,1,8,0
1,3764440,Antonio Barragán Fernández,46,40,86.956522,26,24,92.307692,0,25,0
2,3764440,Clément Lenglet,14,13,92.857143,4,4,100.000000,0,14,0
3,3764440,Edgar Badía Guardiola,38,30,78.947368,5,1,20.000000,0,19,0
4,3764440,Emiliano Ariel Rigoni,23,16,69.565217,18,13,72.222222,0,19,0
5,3764440,Fidel Chaves de la Torre,16,12,75.000000,13,10,76.923077,0,16,0
6,3764440,Francisco António Machado Mota de Castro Trincão,36,34,94.444444,31,29,93.548387,2,37,1
7,3764440,Frenkie de Jong,102,97,95.098039,51,49,96.078431,0,93,0
8,3764440,Gerard Piqué Bernabéu,92,85,92.391304,29,24,82.758621,0,77,0
9,3764440,Gonzalo Cacicedo Verdú,51,51,100.000000,16,16,100.000000,0,43,0


In [221]:
validation_checks = {
    "completed_greater_than_attempted":
        (
            player_match_features["passes_completed"]
            > player_match_features["pass_attempts"]
        ).sum(),

    "pressured_completed_greater_than_attempted":
        (
            player_match_features["pressured_passes_completed"]
            > player_match_features["pressured_pass_attempts"]
        ).sum(),

    "negative_pass_attempts":
        (player_match_features["pass_attempts"] < 0).sum(),

    "completion_rate_above_100":
        (player_match_features["pass_completion_rate"] > 100).sum(),

    "pressure_completion_above_100":
        (
            player_match_features["pressured_pass_completion_rate"] > 100
        ).sum()
}

validation_checks

{'completed_greater_than_attempted': np.int64(0),
 'pressured_completed_greater_than_attempted': np.int64(0),
 'negative_pass_attempts': np.int64(0),
 'completion_rate_above_100': np.int64(0),
 'pressure_completion_above_100': np.int64(0)}

In [222]:
from pathlib import Path

matches_path = Path("open-data-master/data/matches")

print("Exists:", matches_path.exists())

for item in list(matches_path.rglob("*.json"))[:20]:
    print(item)

Exists: True
open-data-master\data\matches\11\1.json
open-data-master\data\matches\11\2.json
open-data-master\data\matches\11\21.json
open-data-master\data\matches\11\22.json
open-data-master\data\matches\11\23.json
open-data-master\data\matches\11\24.json
open-data-master\data\matches\11\25.json
open-data-master\data\matches\11\26.json
open-data-master\data\matches\11\27.json
open-data-master\data\matches\11\278.json
open-data-master\data\matches\11\37.json
open-data-master\data\matches\11\38.json
open-data-master\data\matches\11\39.json
open-data-master\data\matches\11\4.json
open-data-master\data\matches\11\40.json
open-data-master\data\matches\11\41.json
open-data-master\data\matches\11\42.json
open-data-master\data\matches\11\90.json
open-data-master\data\matches\116\68.json
open-data-master\data\matches\12\27.json


In [223]:
import json
import pandas as pd
from pathlib import Path

matches_path = Path("open-data-master/data/matches")

all_match_metadata = []

for file in matches_path.rglob("*.json"):
    try:
        with open(file, "r", encoding="utf-8") as f:
            matches = json.load(f)

        all_match_metadata.extend(matches)

    except Exception as e:
        print(f"Problem with {file}: {e}")

match_metadata = pd.DataFrame(all_match_metadata)

In [224]:
match_metadata[
    [
        "match_id",
        "match_date",
        "home_team",
        "away_team"
    ]
].head(10)

,match_id,match_date,home_team,away_team
0,9880,2018-04-14,"{'home_team_id': 217, 'home_team_name': 'Barce...","{'away_team_id': 207, 'away_team_name': 'Valen..."
1,9912,2018-04-29,"{'home_team_id': 219, 'home_team_name': 'RC De...","{'away_team_id': 217, 'away_team_name': 'Barce..."
2,9924,2018-05-06,"{'home_team_id': 217, 'home_team_name': 'Barce...","{'away_team_id': 220, 'away_team_name': 'Real ..."
3,9855,2018-03-18,"{'home_team_id': 217, 'home_team_name': 'Barce...","{'away_team_id': 215, 'away_team_name': 'Athle..."
4,9827,2018-03-01,"{'home_team_id': 208, 'home_team_name': 'Las P...","{'away_team_id': 217, 'away_team_name': 'Barce..."
5,9799,2018-02-17,"{'home_team_id': 322, 'home_team_name': 'Eibar...","{'away_team_id': 217, 'away_team_name': 'Barce..."
6,9636,2017-10-01,"{'home_team_id': 217, 'home_team_name': 'Barce...","{'away_team_id': 208, 'away_team_name': 'Las P..."
7,9609,2017-09-19,"{'home_team_id': 217, 'home_team_name': 'Barce...","{'away_team_id': 322, 'away_team_name': 'Eibar..."
8,9575,2017-08-20,"{'home_team_id': 217, 'home_team_name': 'Barce...","{'away_team_id': 218, 'away_team_name': 'Real ..."
9,9928,2018-05-09,"{'home_team_id': 217, 'home_team_name': 'Barce...","{'away_team_id': 222, 'away_team_name': 'Villa..."


In [225]:
match_metadata["match_date"] = pd.to_datetime(
    match_metadata["match_date"]
)

print(match_metadata["match_date"].dtype)

datetime64[us]


In [226]:
match_metadata["home_team_name"] = (
    match_metadata["home_team"].apply(
        lambda x: x.get("home_team_name")
        if isinstance(x, dict)
        else None
    )
)

match_metadata["away_team_name"] = (
    match_metadata["away_team"].apply(
        lambda x: x.get("away_team_name")
        if isinstance(x, dict)
        else None
    )
)

In [227]:
match_metadata["home_team_name"] = (
    match_metadata["home_team"].apply(
        lambda x: x.get("home_team_name")
        if isinstance(x, dict)
        else None
    )
)

match_metadata["away_team_name"] = (
    match_metadata["away_team"].apply(
        lambda x: x.get("away_team_name")
        if isinstance(x, dict)
        else None
    )
)

In [228]:
{
    "home_team_id": 217,
    "home_team_name": "Barcelona"
}

{'home_team_id': 217, 'home_team_name': 'Barcelona'}

In [229]:
match_info = match_metadata[
    [
        "match_id",
        "match_date",
        "home_team_name",
        "away_team_name"
    ]
].copy()

In [230]:
match_info.head(10)

,match_id,match_date,home_team_name,away_team_name
0,9880,2018-04-14,Barcelona,Valencia
1,9912,2018-04-29,RC Deportivo La Coruña,Barcelona
2,9924,2018-05-06,Barcelona,Real Madrid
3,9855,2018-03-18,Barcelona,Athletic Club
4,9827,2018-03-01,Las Palmas,Barcelona
5,9799,2018-02-17,Eibar,Barcelona
6,9636,2017-10-01,Barcelona,Las Palmas
7,9609,2017-09-19,Barcelona,Eibar
8,9575,2017-08-20,Barcelona,Real Betis
9,9928,2018-05-09,Barcelona,Villarreal


In [231]:
print("Rows:", len(match_info))
print("Unique match IDs:", match_info["match_id"].nunique())

duplicate_match_ids = match_info[
    match_info["match_id"].duplicated(keep=False)
]

print("Duplicate match IDs:", len(duplicate_match_ids))

Rows: 3961
Unique match IDs: 3961
Duplicate match IDs: 0


In [232]:
print("Rows:", len(match_info))
print("Unique match IDs:", match_info["match_id"].nunique())

duplicate_match_ids = match_info[
    match_info["match_id"].duplicated(keep=False)
]

print("Duplicate match IDs:", len(duplicate_match_ids))

Rows: 3961
Unique match IDs: 3961
Duplicate match IDs: 0


In [234]:
print("Rows:", len(match_info))
print("Unique match IDs:", match_info["match_id"].nunique())

duplicate_match_ids = match_info[
    match_info["match_id"].duplicated(keep=False)
]

print("Duplicate match IDs:", len(duplicate_match_ids))

Rows: 3961
Unique match IDs: 3961
Duplicate match IDs: 0


In [238]:
print("player_match_features match_id dtype:",
      player_match_features["match_id"].dtype)

print("match_info match_id dtype:",
      match_info["match_id"].dtype)

player_match_features match_id dtype: str
match_info match_id dtype: int64


In [239]:
player_match_features["match_id"] = pd.to_numeric(
    player_match_features["match_id"],
    errors="coerce"
)

match_info["match_id"] = pd.to_numeric(
    match_info["match_id"],
    errors="coerce"
)

In [240]:
print(player_match_features["match_id"].dtype)
print(match_info["match_id"].dtype)

int64
int64


In [241]:
player_match_features = player_match_features.merge(
    match_info,
    on="match_id",
    how="left",
    validate="many_to_one"
)

In [242]:
print("Player-match rows:", len(player_match_features))
print(
    "Missing match dates:",
    player_match_features["match_date"].isna().sum()
)

Player-match rows: 13010
Missing match dates: 0


In [243]:
print("Player-match rows:", len(player_match_features))

print(
    "Missing match dates:",
    player_match_features["match_date"].isna().sum()
)

print(
    "Unique matches:",
    player_match_features["match_id"].nunique()
)

Player-match rows: 13010
Missing match dates: 0
Unique matches: 425


In [244]:
player_match_features[
    [
        "match_id",
        "match_date",
        "home_team_name",
        "away_team_name",
        "player_name",
        "pass_attempts",
        "pass_completion_rate"
    ]
].head(20)

,match_id,match_date,home_team_name,away_team_name,player_name,pass_attempts,pass_completion_rate
0,3764440,2021-02-24,Barcelona,Elche,Antoine Griezmann,8,87.500000
1,3764440,2021-02-24,Barcelona,Elche,Antonio Barragán Fernández,46,86.956522
2,3764440,2021-02-24,Barcelona,Elche,Clément Lenglet,14,92.857143
3,3764440,2021-02-24,Barcelona,Elche,Edgar Badía Guardiola,38,78.947368
4,3764440,2021-02-24,Barcelona,Elche,Emiliano Ariel Rigoni,23,69.565217
5,3764440,2021-02-24,Barcelona,Elche,Fidel Chaves de la Torre,16,75.000000
6,3764440,2021-02-24,Barcelona,Elche,Francisco António Machado Mota de Castro Trincão,36,94.444444
7,3764440,2021-02-24,Barcelona,Elche,Frenkie de Jong,102,95.098039
8,3764440,2021-02-24,Barcelona,Elche,Gerard Piqué Bernabéu,92,92.391304
9,3764440,2021-02-24,Barcelona,Elche,Gonzalo Cacicedo Verdú,51,100.000000


In [245]:
print("Player-match rows:", len(player_match_features))

print(
    "Unique matches:",
    player_match_features["match_id"].nunique()
)

print(
    "Missing match dates:",
    player_match_features["match_date"].isna().sum()
)

Player-match rows: 13010
Unique matches: 425
Missing match dates: 0


In [246]:
missing_date_matches = (
    player_match_features.loc[
        player_match_features["match_date"].isna(),
        "match_id"
    ]
    .nunique()
)

print("Matches with missing dates:", missing_date_matches)

Matches with missing dates: 0


In [247]:
print("Player-match rows:", len(player_match_features))

print(
    "Unique matches:",
    player_match_features["match_id"].nunique()
)

print(
    "Missing match dates:",
    player_match_features["match_date"].isna().sum()
)

print(
    "Matches with missing dates:",
    player_match_features.loc[
        player_match_features["match_date"].isna(),
        "match_id"
    ].nunique()
)

Player-match rows: 13010
Unique matches: 425
Missing match dates: 0
Matches with missing dates: 0


In [248]:
def prepare_historical_event_match(match_id):

    events_file = Path(
        f"open-data-master/data/events/{match_id}.json"
    )

    with open(events_file, "r", encoding="utf-8") as f:
        events = json.load(f)

    df = pd.DataFrame(events)

    df["match_id"] = int(match_id)

    df["event_type"] = df["type"].apply(
        lambda x: x.get("name") if isinstance(x, dict) else x
    )

    df["player_name"] = df["player"].apply(
        lambda x: x.get("name") if isinstance(x, dict) else None
    )

    return df

In [249]:
historical_test = prepare_historical_event_match(3764440)

print("Shape:", historical_test.shape)

historical_test[
    [
        "match_id",
        "player_name",
        "event_type"
    ]
].head(20)

Shape: (4160, 39)


,match_id,player_name,event_type
0,3764440,NaN,Starting XI
1,3764440,NaN,Starting XI
2,3764440,NaN,Half Start
3,3764440,NaN,Half Start
4,3764440,Pere Milla Peña,Pass
5,3764440,José Raúl Gutiérrez Parejo,Ball Receipt*
6,3764440,José Raúl Gutiérrez Parejo,Carry
7,3764440,José Raúl Gutiérrez Parejo,Pass
8,3764440,Gonzalo Cacicedo Verdú,Ball Receipt*
9,3764440,Gonzalo Cacicedo Verdú,Carry


In [250]:
historical_test["is_pass"] = (
    historical_test["event_type"] == "Pass"
)

historical_test["is_shot"] = (
    historical_test["event_type"] == "Shot"
)

historical_test["is_carry"] = (
    historical_test["event_type"] == "Carry"
)

historical_test["is_miscontrol"] = (
    historical_test["event_type"] == "Miscontrol"
)

In [251]:
historical_test["pass_completed"] = historical_test["pass"].apply(
    lambda x:
        True if isinstance(x, dict) and x.get("outcome") is None
        else False if isinstance(x, dict)
        else pd.NA
)

In [252]:
historical_test["pass_length"] = historical_test["pass"].apply(
    lambda x: x.get("length")
    if isinstance(x, dict)
    else np.nan
)

historical_test["pass_angle"] = historical_test["pass"].apply(
    lambda x: x.get("angle")
    if isinstance(x, dict)
    else np.nan
)

In [253]:
historical_test["shot_assist"] = historical_test["pass"].apply(
    lambda x: x.get("shot_assist", False)
    if isinstance(x, dict)
    else False
)

In [254]:
historical_test["goal_assist"] = historical_test["pass"].apply(
    lambda x: x.get("goal_assist", False)
    if isinstance(x, dict)
    else False
)


In [255]:
historical_test[
    historical_test["event_type"] == "Pass"
][
    [
        "player_name",
        "pass_completed",
        "pass_length",
        "pass_angle",
        "shot_assist",
        "goal_assist"
    ]
].head(20)

,player_name,pass_completed,pass_length,pass_angle,shot_assist,goal_assist
4,Pere Milla Peña,True,15.597436,2.428794,False,False
7,José Raúl Gutiérrez Parejo,True,17.520845,-2.834281,False,False
10,Gonzalo Cacicedo Verdú,True,27.011848,-1.541175,False,False
13,José Manuel Sánchez Guillén,True,33.664370,1.810732,False,False
16,Gonzalo Cacicedo Verdú,True,55.965080,-0.129009,False,False
21,Marc-André ter Stegen,True,23.493190,-0.994599,False,False
24,Samuel Yves Umtiti,True,33.065086,1.719537,False,False
27,Gerard Piqué Bernabéu,True,15.897799,0.157910,False,False
30,Frenkie de Jong,True,35.203552,-1.585000,False,False
33,Samuel Yves Umtiti,True,22.249720,1.933775,False,False


In [256]:
historical_test["is_pass"] = historical_test["event_type"] == "Pass"
historical_test["is_shot"] = historical_test["event_type"] == "Shot"
historical_test["is_carry"] = historical_test["event_type"] == "Carry"
historical_test["is_miscontrol"] = historical_test["event_type"] == "Miscontrol"

In [257]:
historical_test["pass_completed"] = historical_test["pass"].apply(
    lambda x:
        True if isinstance(x, dict) and x.get("outcome") is None
        else False if isinstance(x, dict)
        else pd.NA
)

In [258]:
historical_test["pass_length"] = historical_test["pass"].apply(
    lambda x: x.get("length") if isinstance(x, dict) else np.nan
)

historical_test["pass_angle"] = historical_test["pass"].apply(
    lambda x: x.get("angle") if isinstance(x, dict) else np.nan
)

historical_test["shot_assist"] = historical_test["pass"].apply(
    lambda x: x.get("shot_assist", False)
    if isinstance(x, dict) else False
)

historical_test["goal_assist"] = historical_test["pass"].apply(
    lambda x: x.get("goal_assist", False)
    if isinstance(x, dict) else False
)

In [259]:
historical_test.loc[
    historical_test["is_pass"],
    [
        "player_name",
        "pass_completed",
        "pass_length",
        "pass_angle",
        "shot_assist",
        "goal_assist"
    ]
].head(20)

,player_name,pass_completed,pass_length,pass_angle,shot_assist,goal_assist
4,Pere Milla Peña,True,15.597436,2.428794,False,False
7,José Raúl Gutiérrez Parejo,True,17.520845,-2.834281,False,False
10,Gonzalo Cacicedo Verdú,True,27.011848,-1.541175,False,False
13,José Manuel Sánchez Guillén,True,33.664370,1.810732,False,False
16,Gonzalo Cacicedo Verdú,True,55.965080,-0.129009,False,False
21,Marc-André ter Stegen,True,23.493190,-0.994599,False,False
24,Samuel Yves Umtiti,True,33.065086,1.719537,False,False
27,Gerard Piqué Bernabéu,True,15.897799,0.157910,False,False
30,Frenkie de Jong,True,35.203552,-1.585000,False,False
33,Samuel Yves Umtiti,True,22.249720,1.933775,False,False


In [260]:
historical_test["pass_angle_degrees"] = np.degrees(
    historical_test["pass_angle"]
)

In [261]:
historical_test.loc[
    historical_test["is_pass"],
    [
        "player_name",
        "pass_length",
        "pass_angle",
        "pass_angle_degrees"
    ]
].head(20)

,player_name,pass_length,pass_angle,pass_angle_degrees
4,Pere Milla Peña,15.597436,2.428794,139.159646
7,José Raúl Gutiérrez Parejo,17.520845,-2.834281,-162.392362
10,Gonzalo Cacicedo Verdú,27.011848,-1.541175,-88.302846
13,José Manuel Sánchez Guillén,33.664370,1.810732,103.747290
16,Gonzalo Cacicedo Verdú,55.965080,-0.129009,-7.391684
21,Marc-André ter Stegen,23.493190,-0.994599,-56.986354
24,Samuel Yves Umtiti,33.065086,1.719537,98.522201
27,Gerard Piqué Bernabéu,15.897799,0.157910,9.047571
30,Frenkie de Jong,35.203552,-1.585000,-90.813805
33,Samuel Yves Umtiti,22.249720,1.933775,110.797163


In [316]:
def prepare_historical_event_match(match_id):

    events_file = Path(
        f"open-data-master/data/events/{match_id}.json"
    )

    with open(events_file, "r", encoding="utf-8") as f:
        events = json.load(f)

    df = pd.DataFrame(events)

    df["match_id"] = int(match_id)

    # --------------------------------------------------
    # Make sure important columns exist
    # --------------------------------------------------

    for column in ["type", "player", "pass", "location"]:
        if column not in df.columns:
            df[column] = None

    # --------------------------------------------------
    # Basic event information
    # --------------------------------------------------

    df["event_type"] = df["type"].apply(
        lambda x: x.get("name")
        if isinstance(x, dict)
        else x
    )

    df["player_name"] = df["player"].apply(
        lambda x: x.get("name")
        if isinstance(x, dict)
        else None
    )

    # --------------------------------------------------
    # Event indicators
    # --------------------------------------------------

    df["is_pass"] = df["event_type"] == "Pass"
    df["is_shot"] = df["event_type"] == "Shot"
    df["is_carry"] = df["event_type"] == "Carry"
    df["is_miscontrol"] = df["event_type"] == "Miscontrol"

    # --------------------------------------------------
    # Pass completion
    # --------------------------------------------------

    df["pass_completed"] = df["pass"].apply(
        lambda x:
            True
            if isinstance(x, dict)
            and x.get("outcome") is None

            else False
            if isinstance(x, dict)

            else pd.NA
    )

    # --------------------------------------------------
    # Pass characteristics
    # --------------------------------------------------

    df["pass_length"] = df["pass"].apply(
        lambda x: x.get("length", np.nan)
        if isinstance(x, dict)
        else np.nan
    )

    df["pass_angle"] = df["pass"].apply(
        lambda x: x.get("angle", np.nan)
        if isinstance(x, dict)
        else np.nan
    )

    # StatsBomb angle is stored in radians.
    # Convert to degrees for easier interpretation.
    df["pass_angle_degrees"] = np.degrees(
        pd.to_numeric(
            df["pass_angle"],
            errors="coerce"
        )
    )

    df["shot_assist"] = df["pass"].apply(
        lambda x: x.get("shot_assist", False)
        if isinstance(x, dict)
        else False
    )

    df["goal_assist"] = df["pass"].apply(
        lambda x: x.get("goal_assist", False)
        if isinstance(x, dict)
        else False
    )

    df["is_cross"] = df["pass"].apply(
        lambda x: x.get("cross", False)
        if isinstance(x, dict)
        else False
    )

    df["is_through_ball"] = df["pass"].apply(
        lambda x: x.get("through_ball", False)
        if isinstance(x, dict)
        else False
    )

    df["is_switch"] = df["pass"].apply(
        lambda x: x.get("switch", False)
        if isinstance(x, dict)
        else False
    )

    # --------------------------------------------------
    # Starting coordinates
    # --------------------------------------------------

    df["start_x"] = df["location"].apply(
        lambda x: x[0]
        if isinstance(x, list) and len(x) >= 2
        else np.nan
    )

    df["start_y"] = df["location"].apply(
        lambda x: x[1]
        if isinstance(x, list) and len(x) >= 2
        else np.nan
    )

    # --------------------------------------------------
    # Safe pass end-location extraction
    # --------------------------------------------------

    def get_pass_end_coordinate(pass_data, index):

        if not isinstance(pass_data, dict):
            return np.nan

        end_location = pass_data.get("end_location")

        if (
            isinstance(end_location, list)
            and len(end_location) >= 2
        ):
            return end_location[index]

        return np.nan

    df["end_x"] = df["pass"].apply(
        lambda x: get_pass_end_coordinate(x, 0)
    )

    df["end_y"] = df["pass"].apply(
        lambda x: get_pass_end_coordinate(x, 1)
    )

    # --------------------------------------------------
    # Forward passing
    # --------------------------------------------------

    df["forward_distance"] = (
        df["end_x"] - df["start_x"]
    )

    df["is_forward_pass"] = (
        df["is_pass"]
        & (df["forward_distance"] > 0)
    )

    # --------------------------------------------------
    # Distance to opponent goal
    # StatsBomb pitch = 120 × 80
    # Opponent goal centre = (120, 40)
    # --------------------------------------------------

    df["distance_to_goal_start"] = np.sqrt(
        (120 - df["start_x"]) ** 2
        + (40 - df["start_y"]) ** 2
    )

    df["distance_to_goal_end"] = np.sqrt(
        (120 - df["end_x"]) ** 2
        + (40 - df["end_y"]) ** 2
    )

    df["goal_distance_reduction"] = (
        df["distance_to_goal_start"]
        - df["distance_to_goal_end"]
    )

    df["goal_distance_reduction_pct"] = np.where(
        df["distance_to_goal_start"] > 0,
        (
            df["goal_distance_reduction"]
            / df["distance_to_goal_start"]
        ) * 100,
        np.nan
    )

    # --------------------------------------------------
    # Progressive pass
    # Working threshold = 25%
    # --------------------------------------------------

    df["is_progressive_pass"] = (
        df["is_pass"]
        & (df["goal_distance_reduction_pct"] >= 25)
    )

    return df

In [317]:
historical_test = prepare_historical_event_match(3764440)

In [318]:
historical_test.loc[
    historical_test["is_pass"],
    [
        "player_name",
        "pass_completed",
        "pass_length",
        "pass_angle_degrees",
        "is_cross",
        "is_through_ball",
        "is_switch",
        "shot_assist",
        "goal_assist"
    ]
].head(20)

,player_name,pass_completed,pass_length,pass_angle_degrees,is_cross,is_through_ball,is_switch,shot_assist,goal_assist
4,Pere Milla Peña,True,15.597436,139.159646,False,False,False,False,False
7,José Raúl Gutiérrez Parejo,True,17.520845,-162.392362,False,False,False,False,False
10,Gonzalo Cacicedo Verdú,True,27.011848,-88.302846,False,False,False,False,False
13,José Manuel Sánchez Guillén,True,33.664370,103.747290,False,False,False,False,False
16,Gonzalo Cacicedo Verdú,True,55.965080,-7.391684,False,False,False,False,False
21,Marc-André ter Stegen,True,23.493190,-56.986354,False,False,False,False,False
24,Samuel Yves Umtiti,True,33.065086,98.522201,False,False,False,False,False
27,Gerard Piqué Bernabéu,True,15.897799,9.047571,False,False,False,False,False
30,Frenkie de Jong,True,35.203552,-90.813805,False,False,False,False,False
33,Samuel Yves Umtiti,True,22.249720,110.797163,False,False,False,False,False


In [319]:
historical_test = prepare_historical_event_match(3764440)

In [320]:
historical_test.loc[
    historical_test["is_pass"],
    [
        "player_name",
        "start_x",
        "start_y",
        "end_x",
        "end_y",
        "forward_distance",
        "is_forward_pass"
    ]
].head(20)

,player_name,start_x,start_y,end_x,end_y,forward_distance,is_forward_pass
4,Pere Milla Peña,61.0,40.1,49.2,50.3,-11.8,False
7,José Raúl Gutiérrez Parejo,50.1,51.4,33.4,46.1,-16.7,False
10,Gonzalo Cacicedo Verdú,34.3,45.3,35.1,18.3,0.8,True
13,José Manuel Sánchez Guillén,34.5,22.8,26.5,55.5,-8.0,False
16,Gonzalo Cacicedo Verdú,34.1,63.3,89.6,56.1,55.5,True
21,Marc-André ter Stegen,12.3,34.6,25.1,14.9,12.8,True
24,Samuel Yves Umtiti,27.3,17.2,22.4,49.9,-4.9,False
27,Gerard Piqué Bernabéu,28.2,53.6,43.9,56.1,15.7,True
30,Frenkie de Jong,43.9,56.1,43.4,20.9,-0.5,False
33,Samuel Yves Umtiti,51.3,19.5,43.4,40.3,-7.9,False


In [321]:
historical_test.loc[
    historical_test["is_pass"],
    [
        "player_name",
        "start_x",
        "start_y",
        "end_x",
        "end_y",
        "distance_to_goal_start",
        "distance_to_goal_end",
        "goal_distance_reduction"
    ]
].head(20)

,player_name,start_x,start_y,end_x,end_y,distance_to_goal_start,distance_to_goal_end,goal_distance_reduction
4,Pere Milla Peña,61.0,40.1,49.2,50.3,59.000085,71.545300,-12.545216
7,José Raúl Gutiérrez Parejo,50.1,51.4,33.4,46.1,70.823513,86.814573,-15.991059
10,Gonzalo Cacicedo Verdú,34.3,45.3,35.1,18.3,85.863729,87.629333,-1.765604
13,José Manuel Sánchez Guillén,34.5,22.8,26.5,55.5,87.212900,94.776052,-7.563151
16,Gonzalo Cacicedo Verdú,34.1,63.3,89.6,56.1,89.003932,34.400145,54.603787
21,Marc-André ter Stegen,12.3,34.6,25.1,14.9,107.835291,98.163231,9.672060
24,Samuel Yves Umtiti,27.3,17.2,22.4,49.9,95.462715,98.100815,-2.638100
27,Gerard Piqué Bernabéu,28.2,53.6,43.9,56.1,92.801940,77.784446,15.017494
30,Frenkie de Jong,43.9,56.1,43.4,20.9,77.784446,78.945361,-1.160915
33,Samuel Yves Umtiti,51.3,19.5,43.4,40.3,71.693375,76.600587,-4.907213


In [322]:
historical_test = prepare_historical_event_match(3764440)

In [323]:
historical_test.loc[
    historical_test["is_pass"],
    [
        "player_name",
        "distance_to_goal_start",
        "distance_to_goal_end",
        "goal_distance_reduction",
        "goal_distance_reduction_pct"
    ]
].head(20)

,player_name,distance_to_goal_start,distance_to_goal_end,goal_distance_reduction,goal_distance_reduction_pct
4,Pere Milla Peña,59.000085,71.545300,-12.545216,-21.263047
7,José Raúl Gutiérrez Parejo,70.823513,86.814573,-15.991059,-22.578744
10,Gonzalo Cacicedo Verdú,85.863729,87.629333,-1.765604,-2.056286
13,José Manuel Sánchez Guillén,87.212900,94.776052,-7.563151,-8.672056
16,Gonzalo Cacicedo Verdú,89.003932,34.400145,54.603787,61.349859
21,Marc-André ter Stegen,107.835291,98.163231,9.672060,8.969290
24,Samuel Yves Umtiti,95.462715,98.100815,-2.638100,-2.763488
27,Gerard Piqué Bernabéu,92.801940,77.784446,15.017494,16.182306
30,Frenkie de Jong,77.784446,78.945361,-1.160915,-1.492477
33,Samuel Yves Umtiti,71.693375,76.600587,-4.907213,-6.844723


In [309]:
passes = historical_test[
    historical_test["is_pass"]
].copy()

for threshold in [20, 25, 30]:
    count = (
        passes["goal_distance_reduction_pct"] >= threshold
    ).sum()

    percentage = (
        count / len(passes) * 100
    )

    print(
        f"{threshold}% threshold: "
        f"{count} progressive passes "
        f"({percentage:.2f}% of all passes)"
    )

20% threshold: 189 progressive passes (15.56% of all passes)
25% threshold: 134 progressive passes (11.03% of all passes)
30% threshold: 107 progressive passes (8.81% of all passes)


In [324]:
historical_player_test = (
    historical_test
    .dropna(subset=["player_name"])
    .groupby(["match_id", "player_name"])
    .agg(
        total_events=("event_type", "count"),

        pass_attempts=("is_pass", "sum"),
        passes_completed=("pass_completed", "sum"),

        forward_passes=("is_forward_pass", "sum"),
        progressive_passes=("is_progressive_pass", "sum"),

        avg_pass_length=("pass_length", "mean"),
        avg_pass_angle_degrees=("pass_angle_degrees", "mean"),

        crosses=("is_cross", "sum"),
        through_balls=("is_through_ball", "sum"),
        switches=("is_switch", "sum"),

        shot_assists=("shot_assist", "sum"),
        goal_assists=("goal_assist", "sum"),

        shots=("is_shot", "sum"),
        carries=("is_carry", "sum"),
        miscontrols=("is_miscontrol", "sum")
    )
    .reset_index()
)

In [325]:
historical_player_test["pass_completion_rate"] = np.where(
    historical_player_test["pass_attempts"] > 0,
    (
        historical_player_test["passes_completed"]
        / historical_player_test["pass_attempts"]
    ) * 100,
    np.nan
)

In [326]:
historical_player_test[
    [
        "player_name",
        "pass_attempts",
        "passes_completed",
        "pass_completion_rate",
        "forward_passes",
        "progressive_passes",
        "shots",
        "carries",
        "miscontrols"
    ]
].sort_values(
    "pass_attempts",
    ascending=False
).head(20)

,player_name,pass_attempts,passes_completed,pass_completion_rate,forward_passes,progressive_passes,shots,carries,miscontrols
7,Frenkie de Jong,102,97,95.098039,65,6,0,93,0
28,Samuel Yves Umtiti,95,90,94.736842,74,3,0,77,0
8,Gerard Piqué Bernabéu,92,85,92.391304,72,5,0,77,0
26,Pedro González López,80,73,91.25,40,5,0,66,1
17,Lionel Andrés Messi Cuccittini,72,60,83.333333,38,20,3,68,2
31,Óscar Mingueza García,71,68,95.774648,29,6,0,59,0
12,Jordi Alba Ramos,70,66,94.285714,35,7,1,51,2
9,Gonzalo Cacicedo Verdú,51,51,100.0,37,9,0,43,0
23,Miralem Pjanić,50,44,88.0,32,7,0,43,0
11,Johan Andrés Mojica Palacio,46,41,89.130435,33,10,0,43,0


In [327]:
print(
    "Completed > attempted:",
    (
        historical_player_test["passes_completed"]
        > historical_player_test["pass_attempts"]
    ).sum()
)

print(
    "Forward passes > attempted:",
    (
        historical_player_test["forward_passes"]
        > historical_player_test["pass_attempts"]
    ).sum()
)

print(
    "Progressive passes > attempted:",
    (
        historical_player_test["progressive_passes"]
        > historical_player_test["pass_attempts"]
    ).sum()
)

print(
    "Completion rate > 100:",
    (
        historical_player_test["pass_completion_rate"] > 100
    ).sum()
)

print(
    "Negative pass attempts:",
    (
        historical_player_test["pass_attempts"] < 0
    ).sum()
)

Completed > attempted: 0
Forward passes > attempted: 0
Progressive passes > attempted: 0
Completion rate > 100: 0
Negative pass attempts: 0


In [328]:
def aggregate_player_match(match_df):

    player_match = (
        match_df
        .dropna(subset=["player_name"])
        .groupby(["match_id", "player_name"])
        .agg(
            total_events=("event_type", "count"),

            pass_attempts=("is_pass", "sum"),
            passes_completed=("pass_completed", "sum"),

            forward_passes=("is_forward_pass", "sum"),
            progressive_passes=("is_progressive_pass", "sum"),

            avg_pass_length=("pass_length", "mean"),
            avg_pass_angle_degrees=("pass_angle_degrees", "mean"),

            crosses=("is_cross", "sum"),
            through_balls=("is_through_ball", "sum"),
            switches=("is_switch", "sum"),

            shot_assists=("shot_assist", "sum"),
            goal_assists=("goal_assist", "sum"),

            shots=("is_shot", "sum"),
            carries=("is_carry", "sum"),
            miscontrols=("is_miscontrol", "sum")
        )
        .reset_index()
    )

    player_match["pass_completion_rate"] = np.where(
        player_match["pass_attempts"] > 0,
        (
            player_match["passes_completed"]
            / player_match["pass_attempts"]
        ) * 100,
        np.nan
    )

    return player_match

In [329]:
historical_player_test_2 = aggregate_player_match(
    historical_test
)

historical_player_test_2[
    [
        "player_name",
        "pass_attempts",
        "passes_completed",
        "pass_completion_rate",
        "forward_passes",
        "progressive_passes",
        "shots",
        "carries",
        "miscontrols"
    ]
].sort_values(
    "pass_attempts",
    ascending=False
).head(10)

,player_name,pass_attempts,passes_completed,pass_completion_rate,forward_passes,progressive_passes,shots,carries,miscontrols
7,Frenkie de Jong,102,97,95.098039,65,6,0,93,0
28,Samuel Yves Umtiti,95,90,94.736842,74,3,0,77,0
8,Gerard Piqué Bernabéu,92,85,92.391304,72,5,0,77,0
26,Pedro González López,80,73,91.25,40,5,0,66,1
17,Lionel Andrés Messi Cuccittini,72,60,83.333333,38,20,3,68,2
31,Óscar Mingueza García,71,68,95.774648,29,6,0,59,0
12,Jordi Alba Ramos,70,66,94.285714,35,7,1,51,2
9,Gonzalo Cacicedo Verdú,51,51,100.0,37,9,0,43,0
23,Miralem Pjanić,50,44,88.0,32,7,0,43,0
11,Johan Andrés Mojica Palacio,46,41,89.130435,33,10,0,43,0


In [330]:
historical_test = prepare_historical_event_match(
    3764440
)

print("Rows:", len(historical_test))
print("Columns:", len(historical_test.columns))

historical_test[
    [
        "player_name",
        "event_type",
        "pass_completed",
        "pass_length",
        "pass_angle_degrees",
        "forward_distance",
        "goal_distance_reduction_pct",
        "is_progressive_pass"
    ]
].head(20)

Rows: 4160
Columns: 63


,player_name,event_type,pass_completed,pass_length,pass_angle_degrees,forward_distance,goal_distance_reduction_pct,is_progressive_pass
0,NaN,Starting XI,<NA>,NaN,NaN,NaN,NaN,False
1,NaN,Starting XI,<NA>,NaN,NaN,NaN,NaN,False
2,NaN,Half Start,<NA>,NaN,NaN,NaN,NaN,False
3,NaN,Half Start,<NA>,NaN,NaN,NaN,NaN,False
4,Pere Milla Peña,Pass,True,15.597436,139.159646,-11.8,-21.263047,False
5,José Raúl Gutiérrez Parejo,Ball Receipt*,<NA>,NaN,NaN,NaN,NaN,False
6,José Raúl Gutiérrez Parejo,Carry,<NA>,NaN,NaN,NaN,NaN,False
7,José Raúl Gutiérrez Parejo,Pass,True,17.520845,-162.392362,-16.7,-22.578744,False
8,Gonzalo Cacicedo Verdú,Ball Receipt*,<NA>,NaN,NaN,NaN,NaN,False
9,Gonzalo Cacicedo Verdú,Carry,<NA>,NaN,NaN,NaN,NaN,False


In [331]:
events_path = Path("open-data-master/data/events")

event_files = list(events_path.glob("*.json"))

print("Events files found:", len(event_files))

Events files found: 4235


In [332]:
historical_player_matches = []
historical_failures = []

for i, events_file in enumerate(event_files, start=1):

    match_id = events_file.stem

    try:
        match_df = prepare_historical_event_match(match_id)

        player_match_df = aggregate_player_match(match_df)

        historical_player_matches.append(player_match_df)

    except Exception as e:

        historical_failures.append(
            {
                "match_id": match_id,
                "error": str(e)
            }
        )

    if i % 500 == 0:
        print(f"Processed {i} / {len(event_files)} matches")

print("Finished")
print("Successful matches:", len(historical_player_matches))
print("Failed matches:", len(historical_failures))

Processed 500 / 4235 matches
Processed 1000 / 4235 matches
Processed 1500 / 4235 matches
Processed 2000 / 4235 matches
Processed 2500 / 4235 matches
Processed 3000 / 4235 matches
Processed 3500 / 4235 matches
Processed 4000 / 4235 matches
Finished
Successful matches: 3439
Failed matches: 796


In [333]:
historical_failures_df = pd.DataFrame(historical_failures)

print(historical_failures_df.shape)

historical_failures_df.head(20)

(796, 2)


,match_id,error
0,15986,division by zero
1,16136,division by zero
2,18240,division by zero
3,18242,division by zero
4,18244,division by zero
5,19714,division by zero
6,19719,division by zero
7,19722,division by zero
8,19738,division by zero
9,19743,division by zero


In [334]:
def aggregate_player_match(match_df):

    player_match = (
        match_df
        .dropna(subset=["player_name"])
        .groupby(["match_id", "player_name"])
        .agg(
            total_events=("event_type", "count"),

            pass_attempts=("is_pass", "sum"),
            passes_completed=("pass_completed", "sum"),

            forward_passes=("is_forward_pass", "sum"),
            progressive_passes=("is_progressive_pass", "sum"),

            avg_pass_length=("pass_length", "mean"),
            avg_pass_angle_degrees=("pass_angle_degrees", "mean"),

            crosses=("is_cross", "sum"),
            through_balls=("is_through_ball", "sum"),
            switches=("is_switch", "sum"),

            shot_assists=("shot_assist", "sum"),
            goal_assists=("goal_assist", "sum"),

            shots=("is_shot", "sum"),
            carries=("is_carry", "sum"),
            miscontrols=("is_miscontrol", "sum")
        )
        .reset_index()
    )

    player_match["pass_attempts"] = pd.to_numeric(
        player_match["pass_attempts"],
        errors="coerce"
    )

    player_match["passes_completed"] = pd.to_numeric(
        player_match["passes_completed"],
        errors="coerce"
    )

    player_match["pass_completion_rate"] = np.nan

    has_pass_attempts = player_match["pass_attempts"] > 0

    player_match.loc[
        has_pass_attempts,
        "pass_completion_rate"
    ] = (
        player_match.loc[
            has_pass_attempts,
            "passes_completed"
        ]
        /
        player_match.loc[
            has_pass_attempts,
            "pass_attempts"
        ]
    ) * 100

    return player_match

In [335]:
problem_test = prepare_historical_event_match(15986)

problem_player_test = aggregate_player_match(
    problem_test
)

print(problem_player_test.shape)

problem_player_test[
    [
        "player_name",
        "pass_attempts",
        "passes_completed",
        "pass_completion_rate"
    ]
]

(28, 18)


,player_name,pass_attempts,passes_completed,pass_completion_rate
0,Aleix García Serrano,18,15,83.333333
1,Alejandro Granell Nogué,41,35,85.365854
2,Arthur Henrique Ramos de Oliveira Melo,65,60,92.307692
3,Arturo Erasmo Vidal Pardo,64,58,90.625000
4,Bernardo José Espinosa Zúñiga,28,23,82.142857
5,Borja García Freire,24,17,70.833333
6,Clément Lenglet,18,18,100.000000
7,Cristhian Ricardo Stuani Curbelo,24,13,54.166667
8,Cristian Portugués Manzanera,13,11,84.615385
9,Douglas Luiz Soares de Paulo,9,8,88.888889


In [336]:
historical_player_matches = []
historical_failures = []

for i, events_file in enumerate(event_files, start=1):

    match_id = events_file.stem

    try:
        match_df = prepare_historical_event_match(match_id)

        player_match_df = aggregate_player_match(match_df)

        historical_player_matches.append(player_match_df)

    except Exception as e:

        historical_failures.append(
            {
                "match_id": match_id,
                "error": str(e)
            }
        )

    if i % 500 == 0:
        print(
            f"Processed {i} / {len(event_files)} matches"
        )

print("\nFinished")
print(
    "Successful matches:",
    len(historical_player_matches)
)
print(
    "Failed matches:",
    len(historical_failures)
)

Processed 500 / 4235 matches
Processed 1000 / 4235 matches
Processed 1500 / 4235 matches
Processed 2000 / 4235 matches
Processed 2500 / 4235 matches
Processed 3000 / 4235 matches
Processed 3500 / 4235 matches
Processed 4000 / 4235 matches

Finished
Successful matches: 4235
Failed matches: 0


In [337]:
historical_player_matches_df = pd.concat(
    historical_player_matches,
    ignore_index=True
)

print("Shape:", historical_player_matches_df.shape)
print(
    "Unique matches:",
    historical_player_matches_df["match_id"].nunique()
)

historical_player_matches_df.head()

Shape: (121034, 18)
Unique matches: 4235


,match_id,player_name,total_events,pass_attempts,passes_completed,forward_passes,progressive_passes,avg_pass_length,avg_pass_angle_degrees,crosses,through_balls,switches,shot_assists,goal_assists,shots,carries,miscontrols,pass_completion_rate
0,15946,Adrián Marín Gómez,29,7,5,5,2,16.931792,47.062771,0,0,0,0,0,1,5,1,71.428571
1,15946,Arthur Henrique Ramos de Oliveira Melo,53,18,17,10,1,20.407317,14.809665,0,1,1,0,1,0,15,0,94.444444
2,15946,Arturo Erasmo Vidal Pardo,22,7,7,1,0,10.627395,40.042062,0,0,0,0,0,0,6,0,100.000000
3,15946,Borja González Tomás,23,6,4,3,0,11.737756,-57.250195,0,0,0,0,0,0,3,0,66.666667
4,15946,Daniel Alejandro Torres Rojas,56,16,12,12,1,15.537846,4.651870,0,0,0,0,0,0,9,0,75.000000


In [338]:
print("Shape:", historical_player_matches_df.shape)

print(
    "Unique matches:",
    historical_player_matches_df["match_id"].nunique()
)

Shape: (121034, 18)
Unique matches: 4235


In [339]:
# Make sure match_id uses the same numeric datatype
historical_player_matches_df["match_id"] = pd.to_numeric(
    historical_player_matches_df["match_id"],
    errors="coerce"
)

match_info["match_id"] = pd.to_numeric(
    match_info["match_id"],
    errors="coerce"
)

# Merge match metadata
historical_with_dates = historical_player_matches_df.merge(
    match_info,
    on="match_id",
    how="left",
    validate="many_to_one"
)

print("Shape after merge:", historical_with_dates.shape)

print(
    "Unique matches:",
    historical_with_dates["match_id"].nunique()
)

print(
    "Player-match rows with missing date:",
    historical_with_dates["match_date"].isna().sum()
)

print(
    "Matches with missing date:",
    historical_with_dates.loc[
        historical_with_dates["match_date"].isna(),
        "match_id"
    ].nunique()
)

Shape after merge: (121034, 21)
Unique matches: 4235
Player-match rows with missing date: 7564
Matches with missing date: 274


In [340]:
historical_with_dates[
    historical_with_dates["match_date"].isna()
][
    ["match_id", "player_name"]
].head(20)

,match_id,player_name
66069,3890259,Albin Ekdal
66070,3890259,Arjen Robben
66071,3890259,Arturo Erasmo Vidal Pardo
66072,3890259,David Olatukunbo Alaba
66073,3890259,Dennis Diekmeier
66074,3890259,Douglas Costa de Souza
66075,3890259,Emir Spahić
66076,3890259,Gideon Jung
66077,3890259,Ivica Olić
66078,3890259,Ivo Iličević


In [341]:
print("Shape after merge:", historical_with_dates.shape)

print(
    "Unique matches:",
    historical_with_dates["match_id"].nunique()
)

print(
    "Player-match rows with missing date:",
    historical_with_dates["match_date"].isna().sum()
)

print(
    "Matches with missing date:",
    historical_with_dates.loc[
        historical_with_dates["match_date"].isna(),
        "match_id"
    ].nunique()
)

Shape after merge: (121034, 21)
Unique matches: 4235
Player-match rows with missing date: 7564
Matches with missing date: 274


In [342]:
missing_date_match_ids = (
    historical_with_dates.loc[
        historical_with_dates["match_date"].isna(),
        "match_id"
    ]
    .drop_duplicates()
    .sort_values()
)

print("Missing-date matches:", len(missing_date_match_ids))

print(
    missing_date_match_ids.head(30).tolist()
)

Missing-date matches: 274
[7298, 69143, 3890259, 3890261, 3890262, 3890263, 3890264, 3890265, 3890266, 3890267, 3890268, 3890269, 3890270, 3890271, 3890272, 3890274, 3890275, 3890276, 3890277, 3890279, 3890280, 3890281, 3890282, 3890283, 3890284, 3890285, 3890286, 3890287, 3890289, 3890290]


In [343]:
print(
    missing_date_match_ids.describe()
)

count    2.740000e+02
mean     3.862293e+06
std      3.285222e+05
min      7.298000e+03
25%      3.890333e+06
50%      3.890410e+06
75%      3.890487e+06
max      3.890564e+06
Name: match_id, dtype: float64


In [344]:
print("Earliest ID:", missing_date_match_ids.min())
print("Latest ID:", missing_date_match_ids.max())

Earliest ID: 7298
Latest ID: 3890564


In [345]:
historical_clean = historical_with_dates.dropna(
    subset=["match_date"]
).copy()

historical_clean["match_date"] = pd.to_datetime(
    historical_clean["match_date"]
)

print("Clean shape:", historical_clean.shape)
print(
    "Unique matches:",
    historical_clean["match_id"].nunique()
)

print(
    "Missing dates:",
    historical_clean["match_date"].isna().sum()
)

Clean shape: (113470, 21)
Unique matches: 3961
Missing dates: 0


In [346]:
validation_results = {
    "duplicate_player_match_rows": historical_clean.duplicated(
        subset=["match_id", "player_name"]
    ).sum(),

    "completed_greater_than_attempted": (
        historical_clean["passes_completed"]
        > historical_clean["pass_attempts"]
    ).sum(),

    "forward_greater_than_attempted": (
        historical_clean["forward_passes"]
        > historical_clean["pass_attempts"]
    ).sum(),

    "progressive_greater_than_attempted": (
        historical_clean["progressive_passes"]
        > historical_clean["pass_attempts"]
    ).sum(),

    "completion_above_100": (
        historical_clean["pass_completion_rate"] > 100
    ).sum(),

    "completion_below_0": (
        historical_clean["pass_completion_rate"] < 0
    ).sum(),

    "negative_pass_attempts": (
        historical_clean["pass_attempts"] < 0
    ).sum(),

    "missing_match_dates": (
        historical_clean["match_date"].isna().sum()
    )
}

for check, result in validation_results.items():
    print(f"{check}: {result}")

duplicate_player_match_rows: 0
completed_greater_than_attempted: 0
forward_greater_than_attempted: 0
progressive_greater_than_attempted: 0
completion_above_100: 0
completion_below_0: 0
negative_pass_attempts: 0
missing_match_dates: 0


In [347]:
historical_clean.to_pickle(
    "historical_player_match_prepared.pkl"
)

print("Historical dataset saved successfully.")

Historical dataset saved successfully.


In [348]:
historical_check = pd.read_pickle(
    "historical_player_match_prepared.pkl"
)

print("Reloaded shape:", historical_check.shape)
print(
    "Reloaded matches:",
    historical_check["match_id"].nunique()
)

print(
    "Missing dates:",
    historical_check["match_date"].isna().sum()
)

Reloaded shape: (113470, 21)
Reloaded matches: 3961
Missing dates: 0


## Player Playing-Time Preparation — Corrected Interval-Based Method

During downstream validation for player-performance forecasting, edge cases were identified in the earlier playing-time calculation. In particular, temporary Player Off/Player On sequences followed by substitutions could produce incorrect exposure estimates. The playing-time preparation was therefore revised using an interval-based approach and revalidated against raw StatsBomb event sequences before downstream feature engineering.


In [1]:
import json
from pathlib import Path
import pandas as pd


# ============================================================
# AUTHORITATIVE PLAYER PLAYING-TIME FUNCTION
# ============================================================

def calculate_player_exposure(match_id, events_dir):

    match_id = str(match_id)
    file_path = Path(events_dir) / f"{match_id}.json"

    with open(file_path, "r", encoding="utf-8") as f:
        events = json.load(f)


    # --------------------------------------------------------
    # 1. StatsBomb match-clock helper
    # --------------------------------------------------------

    def raw_seconds(event):
        return (
            float(event.get("minute", 0)) * 60
            +
            float(event.get("second", 0))
        )


    # --------------------------------------------------------
    # 2. Nominal start clock for each playing period
    #
    # Period 1 = 0:00
    # Period 2 = 45:00
    # Period 3 = 90:00  (extra time)
    # Period 4 = 105:00 (extra time)
    #
    # Period 5 = penalty shoot-out, excluded from exposure
    # --------------------------------------------------------

    nominal_period_start = {
        1: 0,
        2: 45 * 60,
        3: 90 * 60,
        4: 105 * 60
    }


    # --------------------------------------------------------
    # 3. Calculate actual duration of each period
    # --------------------------------------------------------

    period_durations = {}

    for period in [1, 2, 3, 4]:

        period_times = [
            raw_seconds(event)
            for event in events
            if event.get("period") == period
        ]

        if len(period_times) == 0:
            period_durations[period] = 0
            continue

        period_durations[period] = max(
            max(period_times)
            -
            nominal_period_start[period],
            0
        )


    # --------------------------------------------------------
    # 4. Continuous playing-time offsets
    #
    # This removes halftime / extra-time breaks.
    # --------------------------------------------------------

    period_offset = {
        1: 0,
        2: period_durations[1],
        3: (
            period_durations[1]
            +
            period_durations[2]
        ),
        4: (
            period_durations[1]
            +
            period_durations[2]
            +
            period_durations[3]
        )
    }


    def continuous_seconds(event):

        period = event.get("period")

        if period not in [1, 2, 3, 4]:
            return None

        elapsed_in_period = (
            raw_seconds(event)
            -
            nominal_period_start[period]
        )

        elapsed_in_period = max(
            elapsed_in_period,
            0
        )

        return (
            period_offset[period]
            +
            elapsed_in_period
        )


    first_half_end = period_durations[1]

    second_half_end = (
        period_durations[1]
        +
        period_durations[2]
    )

    match_playing_end = sum(
        period_durations.values()
    )


    # --------------------------------------------------------
    # 5. Player state
    # --------------------------------------------------------

    players = {}
    active_players = {}


    def register_player(
        player_id,
        player_name,
        team_name
    ):

        if player_id is None:
            return

        player_id = int(player_id)

        if player_id not in players:

            players[player_id] = {
                "player_id": player_id,
                "player_name": player_name,
                "team_name": team_name,
                "intervals": [],
                "temporary_off_pitch_gaps": 0
            }


    def start_interval(
        player_id,
        time_seconds
    ):

        if player_id is None:
            return

        player_id = int(player_id)

        if player_id not in active_players:

            active_players[player_id] = (
                time_seconds
            )


    def close_interval(
        player_id,
        time_seconds
    ):

        if player_id is None:
            return

        player_id = int(player_id)

        if player_id not in active_players:
            return

        start_time = active_players.pop(
            player_id
        )

        if time_seconds > start_time:

            players[player_id][
                "intervals"
            ].append(
                (
                    start_time,
                    time_seconds
                )
            )


    # --------------------------------------------------------
    # 6. Starting XI
    # --------------------------------------------------------

    starting_xi_events = [
        event
        for event in events
        if event.get("type", {}).get("name")
        == "Starting XI"
    ]


    for event in starting_xi_events:

        team_name = (
            event.get("team", {})
            .get("name")
        )

        lineup = (
            event.get("tactics", {})
            .get("lineup", [])
        )

        for player_info in lineup:

            player = player_info.get(
                "player",
                {}
            )

            player_id = player.get("id")
            player_name = player.get("name")

            register_player(
                player_id,
                player_name,
                team_name
            )

            start_interval(
                player_id,
                0
            )


    # --------------------------------------------------------
    # 7. Sort all playing-period events chronologically
    # --------------------------------------------------------

    playing_events = [
        event
        for event in events
        if event.get("period") in [1, 2, 3, 4]
    ]

    playing_events = sorted(
        playing_events,
        key=lambda event: (
            event.get("period", 0),
            raw_seconds(event)
        )
    )


    # --------------------------------------------------------
    # 8. Process events that change player availability
    # --------------------------------------------------------

    for event in playing_events:

        event_type = (
            event.get("type", {})
            .get("name")
        )

        event_time = continuous_seconds(
            event
        )

        if event_time is None:
            continue


        team_name = (
            event.get("team", {})
            .get("name")
        )


        # ====================================================
        # SUBSTITUTION
        # ====================================================

        if event_type == "Substitution":

            outgoing = event.get(
                "player",
                {}
            )

            outgoing_id = outgoing.get(
                "id"
            )

            close_interval(
                outgoing_id,
                event_time
            )


            replacement = (
                event.get(
                    "substitution",
                    {}
                )
                .get(
                    "replacement",
                    {}
                )
            )

            replacement_id = (
                replacement.get("id")
            )

            replacement_name = (
                replacement.get("name")
            )

            if replacement_id is not None:

                register_player(
                    replacement_id,
                    replacement_name,
                    team_name
                )

                start_interval(
                    replacement_id,
                    event_time
                )


        # ====================================================
        # TEMPORARILY LEAVES PITCH
        # ====================================================

        elif event_type == "Player Off":

            player = event.get(
                "player",
                {}
            )

            player_id = player.get("id")
            player_name = player.get("name")

            if player_id is not None:

                register_player(
                    player_id,
                    player_name,
                    team_name
                )

                close_interval(
                    player_id,
                    event_time
                )


        # ====================================================
        # RETURNS TO PITCH
        # ====================================================

        elif event_type == "Player On":

            player = event.get(
                "player",
                {}
            )

            player_id = player.get("id")
            player_name = player.get("name")

            if player_id is not None:

                register_player(
                    player_id,
                    player_name,
                    team_name
                )

                players[int(player_id)][
                    "temporary_off_pitch_gaps"
                ] += 1

                start_interval(
                    player_id,
                    event_time
                )


        # ====================================================
        # RED CARD / SECOND YELLOW
        # ====================================================

        elif event_type in [
            "Foul Committed",
            "Bad Behaviour"
        ]:

            card_name = None

            if event_type == "Foul Committed":

                card_name = (
                    event.get(
                        "foul_committed",
                        {}
                    )
                    .get(
                        "card",
                        {}
                    )
                    .get("name")
                )

            else:

                card_name = (
                    event.get(
                        "bad_behaviour",
                        {}
                    )
                    .get(
                        "card",
                        {}
                    )
                    .get("name")
                )


            if card_name in [
                "Red Card",
                "Second Yellow"
            ]:

                player_id = (
                    event.get(
                        "player",
                        {}
                    )
                    .get("id")
                )

                close_interval(
                    player_id,
                    event_time
                )


    # --------------------------------------------------------
    # 9. Close everybody still active
    # --------------------------------------------------------

    for player_id in list(
        active_players.keys()
    ):

        close_interval(
            player_id,
            match_playing_end
        )


    # --------------------------------------------------------
    # 10. Merge overlapping intervals
    # --------------------------------------------------------

    def merge_intervals(intervals):

        if not intervals:
            return []

        intervals = sorted(intervals)

        merged = [
            list(intervals[0])
        ]

        for start, end in intervals[1:]:

            previous = merged[-1]

            if start <= previous[1]:

                previous[1] = max(
                    previous[1],
                    end
                )

            else:

                merged.append(
                    [start, end]
                )

        return [
            tuple(interval)
            for interval in merged
        ]


    # --------------------------------------------------------
    # 11. Interval/window overlap
    # --------------------------------------------------------

    def overlap(
        start,
        end,
        window_start,
        window_end
    ):

        return max(
            0,
            min(end, window_end)
            -
            max(start, window_start)
        )


    # --------------------------------------------------------
    # 12. Produce player-level output
    # --------------------------------------------------------

    rows = []


    for player_id, info in players.items():

        intervals = merge_intervals(
            info["intervals"]
        )


        total_seconds = sum(
            end - start
            for start, end in intervals
        )


        first_half_seconds = sum(
            overlap(
                start,
                end,
                0,
                first_half_end
            )
            for start, end in intervals
        )


        second_half_seconds = sum(
            overlap(
                start,
                end,
                first_half_end,
                second_half_end
            )
            for start, end in intervals
        )


        extra_time_seconds = (
            total_seconds
            -
            first_half_seconds
            -
            second_half_seconds
        )


        rows.append(
            {
                "match_id": int(match_id),
                "player_id": player_id,
                "player_name": info[
                    "player_name"
                ],
                "team_name": info[
                    "team_name"
                ],

                "minutes_played":
                    total_seconds / 60,

                "first_half_minutes":
                    first_half_seconds / 60,

                "second_half_minutes":
                    second_half_seconds / 60,

                "extra_time_minutes":
                    extra_time_seconds / 60,

                "temporary_off_pitch_gaps":
                    info[
                        "temporary_off_pitch_gaps"
                    ],

                "match_duration_minutes":
                    match_playing_end / 60,

                "first_half_duration":
                    first_half_end / 60,

                "second_half_duration":
                    period_durations[2] / 60,

                "playing_intervals_seconds":
                    intervals
            }
        )


    return pd.DataFrame(rows)

In [3]:
from pathlib import Path

events_dir = Path("open-data-master") / "data" / "events"

print("Events directory:", events_dir)
print("Exists:", events_dir.exists())

Events directory: open-data-master\data\events
Exists: True


In [4]:
lucas_test = calculate_player_exposure(
    3857279,
    events_dir
)

display(
    lucas_test[
        lucas_test["player_id"] == 5484
    ]
)

,match_id,player_id,player_name,team_name,minutes_played,first_half_minutes,second_half_minutes,extra_time_minutes,temporary_off_pitch_gaps,match_duration_minutes,first_half_duration,second_half_duration,playing_intervals_seconds
4,3857279,5484,Lucas Hernández Pi,France,11.166667,11.166667,0.0,0.0,1,103.45,51.233333,52.216667,"[(0, 670.0)]"


In [5]:
mohamed_test = calculate_player_exposure(
    3920411,
    events_dir
)

display(
    mohamed_test[
        mohamed_test["player_id"] == 70835
    ]
)

,match_id,player_id,player_name,team_name,minutes_played,first_half_minutes,second_half_minutes,extra_time_minutes,temporary_off_pitch_gaps,match_duration_minutes,first_half_duration,second_half_duration,playing_intervals_seconds
23,3920411,70835,Mohamed Hamdy Sharaf,Egypt,57.666667,0.0,57.666667,0.0,0,107.733333,50.066667,57.666667,"[(3004.0, 6464.0)]"


In [6]:
match_id = 3920411
player_id = 70835

file_path = Path(events_dir) / f"{match_id}.json"

with open(file_path, "r", encoding="utf-8") as f:
    events = json.load(f)


availability_events = []

for event in events:

    if event.get("player", {}).get("id") == player_id:

        event_type = event.get("type", {}).get("name")

        if event_type in [
            "Player Off",
            "Player On",
            "Substitution",
            "Foul Committed",
            "Bad Behaviour"
        ]:

            availability_events.append(
                {
                    "period": event.get("period"),
                    "minute": event.get("minute"),
                    "second": event.get("second"),
                    "type": event_type,
                    "player": event.get("player", {}).get("name"),
                    "substitution": event.get("substitution"),
                    "foul_committed": event.get("foul_committed"),
                    "bad_behaviour": event.get("bad_behaviour")
                }
            )

display(pd.DataFrame(availability_events))

,period,minute,second,type,player,substitution,foul_committed,bad_behaviour
0,2,50,22,Foul Committed,Mohamed Hamdy Sharaf,None,None,None


In [7]:
mohamed_match_check = calculate_player_exposure(
    3920411,
    events_dir
)

mohamed_match_check["component_total"] = (
    mohamed_match_check["first_half_minutes"]
    +
    mohamed_match_check["second_half_minutes"]
    +
    mohamed_match_check["extra_time_minutes"]
)

mohamed_match_check["calculation_difference"] = (
    mohamed_match_check["minutes_played"]
    -
    mohamed_match_check["component_total"]
)

print(
    "Players:",
    len(mohamed_match_check)
)

print(
    "Maximum arithmetic difference:",
    mohamed_match_check[
        "calculation_difference"
    ].abs().max()
)

print(
    "Negative minutes:",
    (
        mohamed_match_check[
            "minutes_played"
        ] < 0
    ).sum()
)

print(
    "Players exceeding match duration:",
    (
        mohamed_match_check[
            "minutes_played"
        ]
        >
        mohamed_match_check[
            "match_duration_minutes"
        ]
    ).sum()
)

display(
    mohamed_match_check[
        [
            "player_name",
            "minutes_played",
            "first_half_minutes",
            "second_half_minutes",
            "extra_time_minutes",
            "temporary_off_pitch_gaps",
            "playing_intervals_seconds"
        ]
    ].sort_values(
        "minutes_played",
        ascending=False
    )
)

Players: 32
Maximum arithmetic difference: 1.4210854715202004e-14
Negative minutes: 0
Players exceeding match duration: 0


,player_name,minutes_played,first_half_minutes,second_half_minutes,extra_time_minutes,temporary_off_pitch_gaps,playing_intervals_seconds
0,Josimar José Évora Dias,107.733333,50.066667,57.666667,0.0,0,"[(0, 6464.0)]"
1,Willy Afonso Semedo Johnson,107.733333,50.066667,57.666667,0.0,0,"[(0, 6464.0)]"
2,Logan Costa,107.733333,50.066667,57.666667,0.0,0,"[(0, 6464.0)]"
3,Edilson Alberto Monteiro Sanches Borges,107.733333,50.066667,57.666667,0.0,0,"[(0, 6464.0)]"
4,Dylan Tavares dos Santos,107.733333,50.066667,57.666667,0.0,0,"[(0, 6464.0)]"
7,Kenny Rocha Santos,107.733333,50.066667,57.666667,0.0,0,"[(0, 6464.0)]"
14,Mohamed Abdelmonem,107.733333,50.066667,57.666667,0.0,0,"[(0, 6464.0)]"
13,Ahmed Elsayed Ali Elsayed Hegazy,107.733333,50.066667,57.666667,0.0,0,"[(0, 6464.0)]"
12,Mohamed Hany Gamal Eldemerdash,107.733333,50.066667,57.666667,0.0,0,"[(0, 6464.0)]"
21,Mostafa Mohamed Ahmed Abdallah,107.733333,50.066667,57.666667,0.0,0,"[(0, 6464.0)]"


In [8]:
print("Players:", len(mohamed_match_check))

print(
    "Maximum arithmetic difference:",
    mohamed_match_check["calculation_difference"].abs().max()
)

print(
    "Negative minutes:",
    (mohamed_match_check["minutes_played"] < 0).sum()
)

print(
    "Players exceeding match duration:",
    (
        mohamed_match_check["minutes_played"]
        >
        mohamed_match_check["match_duration_minutes"]
    ).sum()
)

Players: 32
Maximum arithmetic difference: 1.4210854715202004e-14
Negative minutes: 0
Players exceeding match duration: 0


In [9]:
# ============================================================
# BUILD CORRECTED PLAYER EXPOSURE FOR ALL MATCHES
# ============================================================

all_exposure_results = []
failed_matches = []

event_files = sorted(
    Path(events_dir).glob("*.json")
)

total_matches = len(event_files)

print("Total Events files:", total_matches)


for i, file_path in enumerate(event_files, start=1):

    match_id = file_path.stem

    try:

        match_exposure = calculate_player_exposure(
            match_id,
            events_dir
        )

        if not match_exposure.empty:
            all_exposure_results.append(
                match_exposure
            )

    except Exception as e:

        failed_matches.append(
            {
                "match_id": match_id,
                "error": str(e)
            }
        )


    if i % 250 == 0:
        print(
            f"Processed {i}/{total_matches} matches"
        )


corrected_player_minutes = pd.concat(
    all_exposure_results,
    ignore_index=True
)


print("\nPROCESSING COMPLETE")

print(
    "Successful matches:",
    corrected_player_minutes[
        "match_id"
    ].nunique()
)

print(
    "Failed matches:",
    len(failed_matches)
)

print(
    "Rows:",
    len(corrected_player_minutes)
)

print(
    "Unique players:",
    corrected_player_minutes[
        "player_id"
    ].nunique()
)

print(
    "Duplicate match-player rows:",
    corrected_player_minutes.duplicated(
        subset=[
            "match_id",
            "player_id"
        ]
    ).sum()
)


print("\nMinutes summary:")

print(
    corrected_player_minutes[
        [
            "minutes_played",
            "first_half_minutes",
            "second_half_minutes",
            "extra_time_minutes"
        ]
    ].describe()
)

Total Events files: 4235
Processed 250/4235 matches
Processed 500/4235 matches
Processed 750/4235 matches
Processed 1000/4235 matches
Processed 1250/4235 matches
Processed 1500/4235 matches
Processed 1750/4235 matches
Processed 2000/4235 matches
Processed 2250/4235 matches
Processed 2500/4235 matches
Processed 2750/4235 matches
Processed 3000/4235 matches
Processed 3250/4235 matches
Processed 3500/4235 matches
Processed 3750/4235 matches
Processed 4000/4235 matches

PROCESSING COMPLETE
Successful matches: 4235
Failed matches: 0
Rows: 121214
Unique players: 10006
Duplicate match-player rows: 0

Minutes summary:
       minutes_played  first_half_minutes  second_half_minutes  \
count   121214.000000       121214.000000        121214.000000   
mean        74.086194           35.990148            37.785339   
std         30.594617           19.663467            15.381383   
min          0.066667            0.000000             0.000000   
25%         52.450000           44.966667           

In [10]:
print(
    "Negative total minutes:",
    (
        corrected_player_minutes[
            "minutes_played"
        ] < 0
    ).sum()
)

print(
    "Negative first-half minutes:",
    (
        corrected_player_minutes[
            "first_half_minutes"
        ] < 0
    ).sum()
)

print(
    "Negative second-half minutes:",
    (
        corrected_player_minutes[
            "second_half_minutes"
        ] < 0
    ).sum()
)

print(
    "Players exceeding match duration:",
    (
        corrected_player_minutes[
            "minutes_played"
        ]
        >
        corrected_player_minutes[
            "match_duration_minutes"
        ] + 1e-9
    ).sum()
)

component_total = (
    corrected_player_minutes[
        "first_half_minutes"
    ]
    +
    corrected_player_minutes[
        "second_half_minutes"
    ]
    +
    corrected_player_minutes[
        "extra_time_minutes"
    ]
)

print(
    "Maximum component-total difference:",
    (
        corrected_player_minutes[
            "minutes_played"
        ]
        -
        component_total
    ).abs().max()
)

print(
    "Missing player IDs:",
    corrected_player_minutes[
        "player_id"
    ].isna().sum()
)

print(
    "Missing match IDs:",
    corrected_player_minutes[
        "match_id"
    ].isna().sum()
)

Negative total minutes: 0
Negative first-half minutes: 0
Negative second-half minutes: 0
Players exceeding match duration: 0
Maximum component-total difference: 2.842170943040401e-14
Missing player IDs: 0
Missing match IDs: 0


In [11]:
# ============================================================
# COMPARE OLD VS CORRECTED PLAYER MINUTES
# ============================================================

old_minutes = pd.read_pickle(
    "all_player_minutes_prepared.pkl"
).copy()


# Standardise key types
old_minutes["match_id"] = (
    old_minutes["match_id"]
    .astype(str)
)

corrected_player_minutes["match_id"] = (
    corrected_player_minutes["match_id"]
    .astype(str)
)

old_minutes["player_id"] = pd.to_numeric(
    old_minutes["player_id"],
    errors="coerce"
).astype("Int64")

corrected_player_minutes["player_id"] = pd.to_numeric(
    corrected_player_minutes["player_id"],
    errors="coerce"
).astype("Int64")


minutes_comparison = corrected_player_minutes.merge(
    old_minutes[
        [
            "match_id",
            "player_id",
            "minutes_played"
        ]
    ].rename(
        columns={
            "minutes_played":
            "old_minutes_played"
        }
    ),
    on=[
        "match_id",
        "player_id"
    ],
    how="outer",
    indicator=True,
    validate="one_to_one"
)


minutes_comparison[
    "minutes_difference"
] = (
    minutes_comparison[
        "minutes_played"
    ]
    -
    minutes_comparison[
        "old_minutes_played"
    ]
)


print(
    "Comparison rows:",
    len(minutes_comparison)
)

print(
    "\nMerge status:"
)

print(
    minutes_comparison[
        "_merge"
    ].value_counts()
)


matched_rows = minutes_comparison[
    minutes_comparison[
        "_merge"
    ] == "both"
].copy()


print(
    "\nMatched player-match rows:",
    len(matched_rows)
)

print(
    "\nMinutes difference summary:"
)

print(
    matched_rows[
        "minutes_difference"
    ].describe()
)


print(
    "\nAbsolute difference > 1 minute:",
    (
        matched_rows[
            "minutes_difference"
        ].abs() > 1
    ).sum()
)

print(
    "Absolute difference > 5 minutes:",
    (
        matched_rows[
            "minutes_difference"
        ].abs() > 5
    ).sum()
)

print(
    "Absolute difference > 10 minutes:",
    (
        matched_rows[
            "minutes_difference"
        ].abs() > 10
    ).sum()
)

Comparison rows: 161958

Merge status:
_merge
both          121214
right_only     40744
left_only          0
Name: count, dtype: int64

Matched player-match rows: 121214

Minutes difference summary:
count    121214.000000
mean          1.287004
std           3.807097
min        -106.666667
25%           0.000000
50%           1.050000
75%           2.150000
max          73.250000
Name: minutes_difference, dtype: float64

Absolute difference > 1 minute: 67886
Absolute difference > 5 minutes: 6938
Absolute difference > 10 minutes: 1150


In [12]:
display(
    matched_rows[
        [
            "match_id",
            "player_id",
            "player_name",
            "minutes_played",
            "old_minutes_played",
            "minutes_difference",
            "first_half_minutes",
            "second_half_minutes",
            "extra_time_minutes",
            "temporary_off_pitch_gaps"
        ]
    ]
    .assign(
        abs_difference=lambda x:
        x["minutes_difference"].abs()
    )
    .sort_values(
        "abs_difference",
        ascending=False
    )
    .head(30)
)

,match_id,player_id,player_name,minutes_played,old_minutes_played,minutes_difference,first_half_minutes,second_half_minutes,extra_time_minutes,temporary_off_pitch_gaps,abs_difference
147728,4018357,10399,Kathrin Julia Hendrich,10.466667,117.133333,-106.666667,10.466667,0.0,0.0,0.0,106.666667
137185,3913165,131589,Anna Jessica Leat,3.366667,97.716667,-94.350000,3.366667,0.0,0.0,0.0,94.350000
44753,3825569,27122,Simão Mate,3.900000,97.066667,-93.166667,3.900000,0.0,0.0,0.0,93.166667
58131,3829440,3008,Benjamin Lecomte,0.516667,92.950000,-92.433333,0.516667,0.0,0.0,0.0,92.433333
58526,3829451,2989,Alexander Djiku,2.816667,95.200000,-92.383333,2.816667,0.0,0.0,0.0,92.383333
88254,3881590,22027,Anne Moorhouse,2.016667,92.750000,-90.733333,2.016667,0.0,0.0,0.0,90.733333
107410,3900564,26752,Mouhamadou Dabo,2.716667,93.300000,-90.583333,2.716667,0.0,0.0,0.0,90.583333
10318,266149,6576,Gorka Iraizoz Moreno,3.033333,91.983333,-88.950000,3.033333,0.0,0.0,0.0,88.950000
70145,3879561,38237,Gabriel Alejandro Paletta,3.916667,92.833333,-88.916667,3.916667,0.0,0.0,0.0,88.916667
73357,3879634,8180,Leonardo Pavoletti,5.216667,94.066667,-88.850000,5.216667,0.0,0.0,0.0,88.850000


In [13]:
print("Comparison rows:", len(minutes_comparison))

print("\nMerge status:")
print(minutes_comparison["_merge"].value_counts())

matched_rows = minutes_comparison[
    minutes_comparison["_merge"] == "both"
].copy()

print("\nMatched rows:", len(matched_rows))

print("\nDifference summary:")
print(matched_rows["minutes_difference"].describe())

for threshold in [1, 5, 10]:
    print(
        f"Absolute difference > {threshold} minute(s):",
        (
            matched_rows["minutes_difference"].abs()
            > threshold
        ).sum()
    )

Comparison rows: 161958

Merge status:
_merge
both          121214
right_only     40744
left_only          0
Name: count, dtype: int64

Matched rows: 121214

Difference summary:
count    121214.000000
mean          1.287004
std           3.807097
min        -106.666667
25%           0.000000
50%           1.050000
75%           2.150000
max          73.250000
Name: minutes_difference, dtype: float64
Absolute difference > 1 minute(s): 67886
Absolute difference > 5 minute(s): 6938
Absolute difference > 10 minute(s): 1150


In [14]:
old_only = minutes_comparison[
    minutes_comparison["_merge"] == "right_only"
].copy()

print(
    "Old-only rows:",
    len(old_only)
)

print(
    "\nOld minutes summary:"
)

print(
    old_only[
        "old_minutes_played"
    ].describe()
)

print(
    "\nExactly 0 minutes:",
    (
        old_only[
            "old_minutes_played"
        ] == 0
    ).sum()
)

print(
    "Greater than 0 minutes:",
    (
        old_only[
            "old_minutes_played"
        ] > 0
    ).sum()
)

print(
    "Greater than 1 minute:",
    (
        old_only[
            "old_minutes_played"
        ] > 1
    ).sum()
)

print(
    "Greater than 5 minutes:",
    (
        old_only[
            "old_minutes_played"
        ] > 5
    ).sum()
)

Old-only rows: 40744

Old minutes summary:
count    40744.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: old_minutes_played, dtype: float64

Exactly 0 minutes: 40744
Greater than 0 minutes: 0
Greater than 1 minute: 0
Greater than 5 minutes: 0


In [15]:
display(
    old_only[
        old_only[
            "old_minutes_played"
        ] > 0
    ][
        [
            "match_id",
            "player_id",
            "old_minutes_played"
        ]
    ]
    .sort_values(
        "old_minutes_played",
        ascending=False
    )
    .head(30)
)

,match_id,player_id,old_minutes_played


In [16]:
# ============================================================
# SAVE AUTHORITATIVE CORRECTED PLAYER EXPOSURE DATASET
# ============================================================

corrected_player_minutes.to_pickle(
    "all_player_exposure_corrected.pkl"
)

print(
    "Saved: all_player_exposure_corrected.pkl"
)

print(
    "Shape:",
    corrected_player_minutes.shape
)

print(
    "Matches:",
    corrected_player_minutes["match_id"].nunique()
)

print(
    "Players:",
    corrected_player_minutes["player_id"].nunique()
)

print(
    "Duplicate match-player rows:",
    corrected_player_minutes.duplicated(
        subset=["match_id", "player_id"]
    ).sum()
)

Saved: all_player_exposure_corrected.pkl
Shape: (121214, 13)
Matches: 4235
Players: 10006
Duplicate match-player rows: 0
